##### To supress warnings:

In [ ]:
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

### Loading data and Preprocessing

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

# --- Load data ---
train_data = pd.read_excel("2.25Cr1Mo_Train_SelectedFeatures.xlsx")
test_data_10 = pd.read_excel("2.25Cr1Mo_Test_SelectedFeatures(10).xlsx")
val_data_10 = pd.read_excel("2.25Cr1Mo_Validation_SelectedFeatures(10).xlsx")
                            
print(f"Train data shape: {train_data.shape}")
print(f"Test data (10) shape: {test_data_10.shape}")
print(f"Validation (10) shape: {val_data_10.shape}")

features = ['Test Stress', 'T Temp', 'N Temp', 'N', 'Cr', 'Si', 'Test Temp', 'Mo']
X_train = train_data[features]
y_train = train_data['Lg(CL)']  

X_test_10 = test_data_10[features] 
y_test_10 = test_data_10['Lg(CL)']    

X_val_10 = val_data_10[features]
y_val_10 = val_data_10['Lg(CL)'] 

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test_10 shape: {X_test_10.shape}") 
print(f"y_test_10 shape: {y_test_10.shape}")
print(f"X_validation_10 shape: {X_val_10.shape}")
print(f"y_validation_10 shape: {y_val_10.shape}")

# Standardization 
s = StandardScaler()
X_train_s = s.fit_transform(X_train)  
X_test_10_s = s.transform(X_test_10)
X_val_10_s = s.transform(X_val_10)

print("✅ Data loaded and preprocessed successfully!")
print(f"X_train_s shape: {X_train_s.shape}")
print(f"X_test_10_s shape: {X_test_10_s.shape}")
print(f"X_validation_10_s shape: {X_val_10_s.shape}")

### Deep Narrow DNN (3 Hidden Layers)

In [ ]:
# ============================================================
# 🧠 DEEP NARROW DNN (3-LAYER) 
# ============================================================
print("🧠 IMPLEMENTING DEEP NARROW DNN (3-LAYER)")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE DNN 3-LAYER ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================

def create_dnn_3layer_model(input_dim, layers=[32, 24, 16], dropout_rate=0.1, 
                           l2_reg=0.001, learning_rate=0.001):
    """
    Create flexible DNN 3-layer architecture for hyperparameter tuning
    """
    model_dnn_3layer = Sequential()
    
    # Input layer
    model_dnn_3layer.add(Dense(layers[0], activation='relu', input_shape=(input_dim,),
                              kernel_initializer='he_normal',
                              kernel_regularizer=l2(l2_reg)))
    model_dnn_3layer.add(BatchNormalization())
    model_dnn_3layer.add(Dropout(dropout_rate))
    
    # Hidden layers
    for units in layers[1:]:
        model_dnn_3layer.add(Dense(units, activation='relu',
                                  kernel_initializer='he_normal',
                                  kernel_regularizer=l2(l2_reg)))
        model_dnn_3layer.add(BatchNormalization())
        model_dnn_3layer.add(Dropout(dropout_rate * 0.8))  # Slightly less dropout in deeper layers
    
    # Output layer
    model_dnn_3layer.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_dnn_3layer = Adam(learning_rate=learning_rate)
    model_dnn_3layer.compile(optimizer=optimizer_dnn_3layer, loss='mse', metrics=['mae'])
    
    return model_dnn_3layer

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR 3-LAYER DNN...")

# Define hyperparameter combinations for 3-layer DNN
hyperparam_combinations_dnn_3layer = [
    {'layers': [32, 24, 16], 'dropout_rate': 0.1, 'l2_reg': 0.001, 'learning_rate': 0.001},
    {'layers': [48, 32, 16], 'dropout_rate': 0.3, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    {'layers': [24, 16, 8], 'dropout_rate': 0.1, 'l2_reg': 0.0005, 'learning_rate': 0.001},
    {'layers': [64, 32, 16], 'dropout_rate': 0.4, 'l2_reg': 0.005, 'learning_rate': 0.0001},
]

# K-Fold setup
kf_dnn_3layer = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_dnn_3layer = None
best_cv_score_dnn_3layer = -float('inf')
cv_results_dnn_3layer = []

print(f"🔄 Testing {len(hyperparam_combinations_dnn_3layer)} architecture combinations for 3-layer DNN...")

for i, params_dnn_3layer in enumerate(hyperparam_combinations_dnn_3layer):
    print(f"\n🧩 Testing Combination {i+1}: {params_dnn_3layer}")
    
    fold_scores_dnn_3layer = []
    fold_predictions_dnn_3layer = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_dnn_3layer.split(X_train_s)):
        # Create model with current hyperparameters
        model_dnn_3layer_cv = create_dnn_3layer_model(
            input_dim=X_train_s.shape[1],
            layers=params_dnn_3layer['layers'],
            dropout_rate=params_dnn_3layer['dropout_rate'],
            l2_reg=params_dnn_3layer['l2_reg'],
            learning_rate=params_dnn_3layer['learning_rate']
        )
        
        # Callbacks
        early_stopping_dnn_3layer = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_dnn_3layer = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_dnn_3layer_cv = model_dnn_3layer_cv.fit(
            X_train_s[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_s[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_dnn_3layer, reduce_lr_dnn_3layer]
        )
        
        # Get predictions and score
        val_pred_dnn_3layer = model_dnn_3layer_cv.predict(X_train_s[val_idx], verbose=0).flatten()
        fold_r2_dnn_3layer = r2_score(y_train.iloc[val_idx], val_pred_dnn_3layer)
        fold_scores_dnn_3layer.append(fold_r2_dnn_3layer)
        fold_predictions_dnn_3layer[val_idx] = val_pred_dnn_3layer
        
        print(f"   Fold {fold+1} R²: {fold_r2_dnn_3layer:.4f}")
    
    # Calculate overall CV performance
    cv_r2_dnn_3layer = r2_score(y_train, fold_predictions_dnn_3layer)
    mean_fold_r2_dnn_3layer = np.mean(fold_scores_dnn_3layer)
    std_fold_r2_dnn_3layer = np.std(fold_scores_dnn_3layer)
    
    print(f"   📊 CV R²: {cv_r2_dnn_3layer:.4f}, Mean Fold R²: {mean_fold_r2_dnn_3layer:.4f} ± {std_fold_r2_dnn_3layer:.4f}")
    
    # Store results
    result_dnn_3layer = {
        'params': params_dnn_3layer,
        'cv_r2': cv_r2_dnn_3layer,
        'mean_fold_r2': mean_fold_r2_dnn_3layer,
        'std_fold_r2': std_fold_r2_dnn_3layer,
        'fold_scores': fold_scores_dnn_3layer
    }
    cv_results_dnn_3layer.append(result_dnn_3layer)
    
    # Update best parameters
    if cv_r2_dnn_3layer > best_cv_score_dnn_3layer:
        best_cv_score_dnn_3layer = cv_r2_dnn_3layer
        best_params_dnn_3layer = params_dnn_3layer
        best_cv_predictions_dnn_3layer = fold_predictions_dnn_3layer.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR 3-LAYER DNN:")
print(f"Layers: {best_params_dnn_3layer['layers']}")
print(f"Dropout: {best_params_dnn_3layer['dropout_rate']}")
print(f"L2 Reg: {best_params_dnn_3layer['l2_reg']}")
print(f"Learning Rate: {best_params_dnn_3layer['learning_rate']}")
print(f"Best CV R²: {best_cv_score_dnn_3layer:.4f}")

# ============================================================
# TRAIN FINAL 3-LAYER MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL 3-LAYER DNN WITH BEST PARAMETERS...")

# Create final model with best parameters
dnn_3layer_model = create_dnn_3layer_model(
    input_dim=X_train_s.shape[1],
    layers=best_params_dnn_3layer['layers'],
    dropout_rate=best_params_dnn_3layer['dropout_rate'],
    l2_reg=best_params_dnn_3layer['l2_reg'],
    learning_rate=best_params_dnn_3layer['learning_rate']
)

print(f"✅ FINAL 3-LAYER DNN ARCHITECTURE: 8 → {' → '.join(map(str, best_params_dnn_3layer['layers']))} → 1")
dnn_3layer_model.summary()

# Final training callbacks for 3-layer DNN
early_stopping_dnn_3layer = EarlyStopping(
    monitor='loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_dnn_3layer = ReduceLROnPlateau(
    monitor='loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final 3-layer model on full training data with validation set
history_dnn_3layer = dnn_3layer_model.fit(
    X_train_s, y_train,
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_dnn_3layer, reduce_lr_dnn_3layer],
    shuffle=True
)

# Then evaluate on validation set separately
print("📊 EVALUATING ON VALIDATION SET...")
val_loss, val_mae = dnn_3layer_model.evaluate(X_val_10_s, y_val_10, verbose=0)
print(f"✅ Validation Loss: {val_loss:.4f}, Validation MAE: {val_mae:.4f}")


# ============================================================
# PREDICTIONS AND METRICS FOR 3-LAYER DNN
# ============================================================
print("📊 EVALUATING FINAL 3-LAYER DNN MODEL...")

# Predictions
y_pred_log_train_dnn_3layer = dnn_3layer_model.predict(X_train_s).flatten()
y_pred_log_val_10_dnn_3layer = dnn_3layer_model.predict(X_val_10_s).flatten()
y_pred_log_test_10_dnn_3layer = dnn_3layer_model.predict(X_test_10_s).flatten()

# Metrics function for 3-layer DNN
def compute_metrics_dnn_3layer(y_true, y_pred):
    r2_dnn_3layer = r2_score(y_true, y_pred)
    mse_dnn_3layer = mean_squared_error(y_true, y_pred)
    mae_dnn_3layer = mean_absolute_error(y_true, y_pred)
    return r2_dnn_3layer, mse_dnn_3layer, mae_dnn_3layer

# Calculate metrics for 3-layer DNN
r2_dnn_train_3layer, mse_dnn_train_3layer, mae_dnn_train_3layer = compute_metrics_dnn_3layer(y_train, y_pred_log_train_dnn_3layer)
r2_dnn_val_10_3layer, mse_dnn_val_10_3layer, mae_dnn_val_10_3layer = compute_metrics_dnn_3layer(y_val_10, y_pred_log_val_10_dnn_3layer)
r2_dnn_test_10_3layer, mse_dnn_test_10_3layer, mae_dnn_test_10_3layer = compute_metrics_dnn_3layer(y_test_10, y_pred_log_test_10_dnn_3layer)

# ============================================================
# COMPREHENSIVE RESULTS FOR 3-LAYER DNN
# ============================================================
print("\n📊 3-LAYER DNN PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_dnn_train_3layer:.4f}")
print(f"Validation R² (log): {r2_dnn_val_10_3layer:.4f}")
print(f"Testing R² (log):    {r2_dnn_test_10_3layer:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_dnn_3layer:.4f}")

print(f"\nCross-Validation Fold R² Scores for 3-Layer DNN:")
for i, result_dnn_3layer in enumerate(cv_results_dnn_3layer):
    if result_dnn_3layer['params'] == best_params_dnn_3layer:
        print(f"  🏆 Best 3-Layer Model Fold R²: {[f'{score:.4f}' for score in result_dnn_3layer['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_dnn_train_3layer:.4f}")
print(f"Validation MSE (log): {mse_dnn_val_10_3layer:.4f}")
print(f"Testing MSE (log):   {mse_dnn_test_10_3layer:.4f}")

print(f"\nTraining MAE (log):   {mae_dnn_train_3layer:.4f}")
print(f"Validation MAE (log): {mae_dnn_val_10_3layer:.4f}")
print(f"Testing MAE (log):    {mae_dnn_test_10_3layer:.4f}")

print(f"\nR² Difference (Train - Test): {r2_dnn_train_3layer - r2_dnn_test_10_3layer:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_dnn_train_3layer - r2_dnn_test_10_3layer) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR 3-LAYER DNN
# ============================================================
y_pred_train_dnn_3layer = np.exp(y_pred_log_train_dnn_3layer)
y_train_actual = np.exp(y_train)
y_pred_test_10_dnn_3layer = np.exp(y_pred_log_test_10_dnn_3layer)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_dnn_3layer = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_dnn_3layer[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_dnn_3layer[:10]
}).round(4)
print(comparison_log_df_dnn_3layer)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_dnn_3layer = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_dnn_3layer[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_dnn_3layer[:10]
}).round(2)
print(comparison_actual_df_dnn_3layer)

# ============================================================
# HYPERPARAMETER COMPARISON FOR 3-LAYER DNN
# ============================================================
print("\n🔍 3-LAYER DNN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_dnn_3layer in enumerate(cv_results_dnn_3layer):
    marker = "🏆" if result_dnn_3layer['params'] == best_params_dnn_3layer else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_dnn_3layer['cv_r2']:.4f}, "
          f"Layers = {result_dnn_3layer['params']['layers']}, "
          f"Dropout = {result_dnn_3layer['params']['dropout_rate']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR 3-LAYER DNN
# ============================================================
print("\n📈 PLOTTING 3-LAYER DNN TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_dnn_3layer.history['loss'], label='Training Loss')
plt.title('3-Layer DNN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_dnn_3layer.history['mae'], label='Training MAE')
plt.title('3-Layer DNN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ DEEP NARROW DNN (3-LAYER) WITH K-FOLD CV COMPLETED!")
print("="*60)

### Plots DNN (3 Layer) 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_dnn_3layer = y_train - y_pred_log_train_dnn_3layer
test_residuals_dnn_3layer = y_test_10 - y_pred_log_test_10_dnn_3layer

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_dnn_3layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_dnn_3layer, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("DNN (3 layer) : Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_dnn_3layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_dnn_3layer, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("DNN (3 layer): Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_dnn_3layer, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_dnn_3layer, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("DNN (3 layer): Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 SCATTER BAND ANALYSIS: DNN (3 Hidden Layers)

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: DNN (3 Layer)
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR DNN (3 Layer)...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_dnn_3layer = y_pred_test_10_dnn_3layer / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_dnn_3layer = np.mean((scatter_ratio_dnn_3layer >= 0.5) & (scatter_ratio_dnn_3layer <= 2.0)) * 100
within_1_5x_dnn_3layer = np.mean((scatter_ratio_dnn_3layer >= 0.67) & (scatter_ratio_dnn_3layer <= 1.5)) * 100
within_3x_dnn_3layer = np.mean((scatter_ratio_dnn_3layer >= 0.33) & (scatter_ratio_dnn_3layer <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_dnn_3layer = np.mean(scatter_ratio_dnn_3layer < 1.0) * 100  # Under-predictions
optimistic_dnn_3layer = np.mean(scatter_ratio_dnn_3layer > 1.0) * 100    # Over-predictions

print(f"📊 DNN (3 Layer) SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_dnn_3layer:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_dnn_3layer:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_dnn_3layer:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_dnn_3layer:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_dnn_3layer:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_dnn_3layer, alpha=0.7, color='blue', s=50, 
           label=f'DNN (3 Layer) Predictions ({within_2x_dnn_3layer:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_dnn_3layer.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_dnn_3layer.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('DNN (3 Layer): ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_dnn_3layer, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_dnn_3layer)
mean_ratio = np.mean(scatter_ratio_dnn_3layer)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'DNN (3 Layer): Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_dnn_3layer:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_dnn_3layer)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_dnn_3layer):.2f}
Within ±2: {within_2x_dnn_3layer:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### One by one predictions

In [ ]:
# ============================================================
# 📊 PREDICTIONS FOR ANALYSIS
# ============================================================
print("📊 EXTRACTING PREDICTION VALUES...")

# Get predictions in actual scale
y_pred_test_10_actual = np.exp(y_pred_log_test_10_dnn_3layer)
y_test_10_actual = np.exp(y_test_10)

# Calculate percentage errors
percentage_errors = ((y_test_10_actual - y_pred_test_10_actual) / y_test_10_actual) * 100

# Create simple comparison table
comparison_df = pd.DataFrame({
    'Actual': y_test_10_actual,
    'Predicted': y_pred_test_10_actual,
    'Error_%': percentage_errors
}).round(2)

print("PREDICTIONS VS ACTUAL VALUES:")
print(comparison_df)

### Deep Narrow DNN (2 Hidden Layers)

In [ ]:
# ============================================================
# 🧠 DEEP NARROW DNN (2-LAYER) 
# ============================================================
print("🧠 IMPLEMENTING DEEP NARROW DNN (2-LAYER)")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE DNN 2-LAYER ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_dnn_2layer_model(input_dim, layers=[64, 48], dropout_rate=0.1, 
                           l2_reg=0.001, learning_rate=0.001):
    """
    Create flexible DNN 2-layer architecture for hyperparameter tuning
    """
    model_dnn_2layer = Sequential()
    
    # Input layer
    model_dnn_2layer.add(Dense(layers[0], activation='relu', input_shape=(input_dim,),
                              kernel_initializer='he_normal',
                              kernel_regularizer=l2(l2_reg)))
    model_dnn_2layer.add(BatchNormalization())
    model_dnn_2layer.add(Dropout(dropout_rate))
    
    # Hidden layer
    model_dnn_2layer.add(Dense(layers[1], activation='relu',
                              kernel_initializer='he_normal',
                              kernel_regularizer=l2(l2_reg)))
    model_dnn_2layer.add(BatchNormalization())
    model_dnn_2layer.add(Dropout(dropout_rate * 0.8))
    
    # Output layer
    model_dnn_2layer.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_dnn_2layer = Adam(learning_rate=learning_rate)
    model_dnn_2layer.compile(optimizer=optimizer_dnn_2layer, loss='mse', metrics=['mae'])
    
    return model_dnn_2layer

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR 2-LAYER DNN...")

# Define hyperparameter combinations for 2-layer DNN
hyperparam_combinations_dnn_2layer = [
    {'layers': [64, 48], 'dropout_rate': 0.4, 'l2_reg': 0.005, 'learning_rate': 0.0001},
]

# K-Fold setup
kf_dnn_2layer = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_dnn_2layer = None
best_cv_score_dnn_2layer = -float('inf')
cv_results_dnn_2layer = []

print(f"🔄 Testing {len(hyperparam_combinations_dnn_2layer)} architecture combinations for 2-layer DNN...")

for i, params_dnn_2layer in enumerate(hyperparam_combinations_dnn_2layer):
    print(f"\n🧩 Testing Combination {i+1}: {params_dnn_2layer}")
    
    fold_scores_dnn_2layer = []
    fold_predictions_dnn_2layer = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_dnn_2layer.split(X_train_s)):
        # Create model with current hyperparameters
        model_dnn_2layer_cv = create_dnn_2layer_model(
            input_dim=X_train_s.shape[1],
            layers=params_dnn_2layer['layers'],
            dropout_rate=params_dnn_2layer['dropout_rate'],
            l2_reg=params_dnn_2layer['l2_reg'],
            learning_rate=params_dnn_2layer['learning_rate']
        )
        
        # Callbacks
        early_stopping_dnn_2layer = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_dnn_2layer = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_dnn_2layer_cv = model_dnn_2layer_cv.fit(
            X_train_s[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_s[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_dnn_2layer, reduce_lr_dnn_2layer]
        )
        
        # Get predictions and score
        val_pred_dnn_2layer = model_dnn_2layer_cv.predict(X_train_s[val_idx], verbose=0).flatten()
        fold_r2_dnn_2layer = r2_score(y_train.iloc[val_idx], val_pred_dnn_2layer)
        fold_scores_dnn_2layer.append(fold_r2_dnn_2layer)
        fold_predictions_dnn_2layer[val_idx] = val_pred_dnn_2layer
        
        print(f"   Fold {fold+1} R²: {fold_r2_dnn_2layer:.4f}")
    
    # Calculate overall CV performance
    cv_r2_dnn_2layer = r2_score(y_train, fold_predictions_dnn_2layer)
    mean_fold_r2_dnn_2layer = np.mean(fold_scores_dnn_2layer)
    std_fold_r2_dnn_2layer = np.std(fold_scores_dnn_2layer)
    
    print(f"   📊 CV R²: {cv_r2_dnn_2layer:.4f}, Mean Fold R²: {mean_fold_r2_dnn_2layer:.4f} ± {std_fold_r2_dnn_2layer:.4f}")
    
    # Store results
    result_dnn_2layer = {
        'params': params_dnn_2layer,
        'cv_r2': cv_r2_dnn_2layer,
        'mean_fold_r2': mean_fold_r2_dnn_2layer,
        'std_fold_r2': std_fold_r2_dnn_2layer,
        'fold_scores': fold_scores_dnn_2layer
    }
    cv_results_dnn_2layer.append(result_dnn_2layer)
    
    # Update best parameters
    if cv_r2_dnn_2layer > best_cv_score_dnn_2layer:
        best_cv_score_dnn_2layer = cv_r2_dnn_2layer
        best_params_dnn_2layer = params_dnn_2layer
        best_cv_predictions_dnn_2layer = fold_predictions_dnn_2layer.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR 2-LAYER DNN:")
print(f"Layers: {best_params_dnn_2layer['layers']}")
print(f"Dropout: {best_params_dnn_2layer['dropout_rate']}")
print(f"L2 Reg: {best_params_dnn_2layer['l2_reg']}")
print(f"Learning Rate: {best_params_dnn_2layer['learning_rate']}")
print(f"Best CV R²: {best_cv_score_dnn_2layer:.4f}")

# ============================================================
# TRAIN FINAL 2-LAYER MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL 2-LAYER DNN WITH BEST PARAMETERS...")

# Create final model with best parameters
dnn_2layer_model = create_dnn_2layer_model(
    input_dim=X_train_s.shape[1],
    layers=best_params_dnn_2layer['layers'],
    dropout_rate=best_params_dnn_2layer['dropout_rate'],
    l2_reg=best_params_dnn_2layer['l2_reg'],
    learning_rate=best_params_dnn_2layer['learning_rate']
)

print(f"✅ FINAL 2-LAYER DNN ARCHITECTURE: 8 → {' → '.join(map(str, best_params_dnn_2layer['layers']))} → 1")
dnn_2layer_model.summary()

# Final training callbacks for 2-layer DNN
early_stopping_dnn_2layer = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_dnn_2layer = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final 2-layer model on full training data with validation set
history_dnn_2layer = dnn_2layer_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_10_s, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_dnn_2layer, reduce_lr_dnn_2layer],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR 2-LAYER DNN
# ============================================================
print("📊 EVALUATING FINAL 2-LAYER DNN MODEL...")

# Predictions
y_pred_log_train_dnn_2layer = dnn_2layer_model.predict(X_train_s).flatten()
y_pred_log_val_10_dnn_2layer = dnn_2layer_model.predict(X_val_10_s).flatten()
y_pred_log_test_10_dnn_2layer = dnn_2layer_model.predict(X_test_10_s).flatten()

# Metrics function for 2-layer DNN
def compute_metrics_dnn_2layer(y_true, y_pred):
    r2_dnn_2layer = r2_score(y_true, y_pred)
    mse_dnn_2layer = mean_squared_error(y_true, y_pred)
    mae_dnn_2layer = mean_absolute_error(y_true, y_pred)
    return r2_dnn_2layer, mse_dnn_2layer, mae_dnn_2layer

# Calculate metrics for 2-layer DNN
r2_dnn_train_2layer, mse_dnn_train_2layer, mae_dnn_train_2layer = compute_metrics_dnn_2layer(y_train, y_pred_log_train_dnn_2layer)
r2_dnn_val_10_2layer, mse_dnn_val_10_2layer, mae_dnn_val_10_2layer = compute_metrics_dnn_2layer(y_val_10, y_pred_log_val_10_dnn_2layer)
r2_dnn_test_10_2layer, mse_dnn_test_10_2layer, mae_dnn_test_10_2layer = compute_metrics_dnn_2layer(y_test_10, y_pred_log_test_10_dnn_2layer)

# ============================================================
# COMPREHENSIVE RESULTS FOR 2-LAYER DNN
# ============================================================
print("\n📊 2-LAYER DNN PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_dnn_train_2layer:.4f}")
print(f"Validation R² (log): {r2_dnn_val_10_2layer:.4f}")
print(f"Testing R² (log):    {r2_dnn_test_10_2layer:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_dnn_2layer:.4f}")

print(f"\nCross-Validation Fold R² Scores for 2-Layer DNN:")
for i, result_dnn_2layer in enumerate(cv_results_dnn_2layer):
    if result_dnn_2layer['params'] == best_params_dnn_2layer:
        print(f"  🏆 Best 2-Layer Model Fold R²: {[f'{score:.4f}' for score in result_dnn_2layer['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_dnn_train_2layer:.4f}")
print(f"Validation MSE (log): {mse_dnn_val_10_2layer:.4f}")
print(f"Testing MSE (log):   {mse_dnn_test_10_2layer:.4f}")

print(f"\nTraining MAE (log):   {mae_dnn_train_2layer:.4f}")
print(f"Validation MAE (log): {mae_dnn_val_10_2layer:.4f}")
print(f"Testing MAE (log):    {mae_dnn_test_10_2layer:.4f}")

print(f"\nR² Difference (Train - Test): {r2_dnn_train_2layer - r2_dnn_test_10_2layer:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_dnn_train_2layer - r2_dnn_test_10_2layer) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR 2-LAYER DNN
# ============================================================
y_pred_train_dnn_2layer = np.exp(y_pred_log_train_dnn_2layer)
y_train_actual = np.exp(y_train)
y_pred_test_10_dnn_2layer = np.exp(y_pred_log_test_10_dnn_2layer)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_dnn_2layer = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_dnn_2layer[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_dnn_2layer[:10]
}).round(4)
print(comparison_log_df_dnn_2layer)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_dnn_2layer = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_dnn_2layer[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_dnn_2layer[:10]
}).round(2)
print(comparison_actual_df_dnn_2layer)

# ============================================================
# HYPERPARAMETER COMPARISON FOR 2-LAYER DNN
# ============================================================
print("\n🔍 2-LAYER DNN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_dnn_2layer in enumerate(cv_results_dnn_2layer):
    marker = "🏆" if result_dnn_2layer['params'] == best_params_dnn_2layer else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_dnn_2layer['cv_r2']:.4f}, "
          f"Layers = {result_dnn_2layer['params']['layers']}, "
          f"Dropout = {result_dnn_2layer['params']['dropout_rate']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR 2-LAYER DNN
# ============================================================
print("\n📈 PLOTTING 2-LAYER DNN TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_dnn_2layer.history['loss'], label='Training Loss')
plt.plot(history_dnn_2layer.history['val_loss'], label='Validation Loss')
plt.title('2-Layer DNN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_dnn_2layer.history['mae'], label='Training MAE')
plt.plot(history_dnn_2layer.history['val_mae'], label='Validation MAE')
plt.title('2-Layer DNN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ DEEP NARROW DNN (2-LAYER) WITH K-FOLD CV COMPLETED!")
print("="*60)

### PLots DNN (2 Layer)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_dnn_2layer = y_train - y_pred_log_train_dnn_2layer
test_residuals_dnn_2layer = y_test_10 - y_pred_log_test_10_dnn_2layer

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_dnn_2layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_dnn_2layer, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("DNN (2 layer) : Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_dnn_2layer, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_dnn_2layer, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("DNN (2 layer): Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_dnn_2layer, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_dnn_2layer, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("DNN (2 layer): Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 SCATTER BAND ANALYSIS: DNN (2 Hidden Layers)

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: DNN (2 Layer)
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR DNN (2 Layer)...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_dnn_2layer = y_pred_test_10_dnn_2layer / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_dnn_2layer = np.mean((scatter_ratio_dnn_2layer >= 0.5) & (scatter_ratio_dnn_2layer <= 2.0)) * 100
within_1_5x_dnn_2layer = np.mean((scatter_ratio_dnn_2layer >= 0.67) & (scatter_ratio_dnn_2layer <= 1.5)) * 100
within_3x_dnn_2layer = np.mean((scatter_ratio_dnn_2layer >= 0.33) & (scatter_ratio_dnn_2layer <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_dnn_2layer = np.mean(scatter_ratio_dnn_2layer < 1.0) * 100  # Under-predictions
optimistic_dnn_2layer = np.mean(scatter_ratio_dnn_2layer > 1.0) * 100    # Over-predictions

print(f"📊 DNN (2 Layer) SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_dnn_2layer:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_dnn_2layer:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_dnn_2layer:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_dnn_2layer:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_dnn_2layer:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_dnn_2layer, alpha=0.7, color='blue', s=50, 
           label=f'DNN (2 Layer) Predictions ({within_2x_dnn_2layer:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_dnn_2layer.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_dnn_2layer.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('DNN (2 Layer): ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_dnn_2layer, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_dnn_2layer)
mean_ratio = np.mean(scatter_ratio_dnn_2layer)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'DNN (2 Layer): Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_dnn_2layer:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_dnn_2layer)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_dnn_2layer):.2f}
Within ±2: {within_2x_dnn_2layer:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### Bayesian-Style Neural Network  

In [ ]:
# ============================================================
# 🧠 BAYESIAN-STYLE NEURAL NETWORK 
# ============================================================
print("🧠 IMPLEMENTING BAYESIAN-STYLE NEURAL NETWORK WITH K-FOLD CV...")
print("="*60)

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import random

def set_seeds(seed=42):
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    
set_seeds(42)

# ============================================================
# FLEXIBLE BAYESIAN-STYLE ARCHITECTURE
# ============================================================
def create_bayesian_style_model(input_dim, layers=[24, 16, 8], dropout_rates=[0.08, 0.04, 0.02], 
                               learning_rate=0.001):
    """
    Create flexible Bayesian-style neural network with dropout for uncertainty
    """
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(input_dim,)))
    
    # Add layers with specified dropout rates
    for i, (units, dropout_rate) in enumerate(zip(layers, dropout_rates)):
        model.add(tf.keras.layers.Dense(units, activation='relu', kernel_initializer='he_normal'))
        model.add(tf.keras.layers.Dropout(dropout_rate))
    
    # Output layer
    model.add(tf.keras.layers.Dense(1, activation=None))
    
    optimizer = Adam(learning_rate=learning_rate,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-07, 
        clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR BAYESIAN-STYLE NN...")

# Define hyperparameter combinations
hyperparam_combinations_bnn = [
    {'layers': [48, 32, 24], 'dropout_rates': [0.3, 0.15, 0.1], 'learning_rate': 0.0005},
    {'layers': [48, 32, 16], 'dropout_rates': [0.3, 0.15, 0.08], 'learning_rate': 0.0005},
    {'layers': [32, 24, 16], 'dropout_rates': [0.15, 0.08, 0.04], 'learning_rate': 0.0001},
    {'layers': [24, 16, 8], 'dropout_rates': [0.08, 0.04, 0.02], 'learning_rate': 0.001},
]

# K-Fold setup
kf_bnn = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_bnn = None
best_cv_score_bnn = -float('inf')
cv_results_bnn = []

print(f"🔄 Testing {len(hyperparam_combinations_bnn)} Bayesian-style architecture combinations...")

for i, params_bnn in enumerate(hyperparam_combinations_bnn):
    print(f"\n🧩 Testing Combination {i+1}: {params_bnn}")
    
    fold_scores_bnn = []
    fold_predictions_bnn = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_bnn.split(X_train_s)):
        # Create model with current hyperparameters
        model_bnn_cv = create_bayesian_style_model(
            input_dim=X_train_s.shape[1],
            layers=params_bnn['layers'],
            dropout_rates=params_bnn['dropout_rates'],
            learning_rate=params_bnn['learning_rate']
        )
        
        # Callbacks
        early_stopping_bnn = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_bnn = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_bnn_cv = model_bnn_cv.fit(
            X_train_s[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_s[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=16,
            verbose=0,
            callbacks=[early_stopping_bnn, reduce_lr_bnn]
        )
        
        # ============================================================
        # QUICK PREDICTION FOR CV (SPEED OPTIMIZATION)
        # ============================================================
        def predict_quick_fold(model, X):

            """
            Fast prediction for CV (no MC sampling to save time)
            """

            return model.predict(X, verbose=0).flatten()
        
        
        def predict_mc_fold(model, X, n_samples=100):

            """
            Full MC Dropout only for final evaluation
            """
            preds = np.array([model(X, training=True).numpy().flatten() for _ in range(n_samples)])
            return preds.mean(axis=0),  preds.std(axis=0)
        
        val_pred_bnn = predict_quick_fold(model_bnn_cv, X_train_s[val_idx])
        fold_r2_bnn = r2_score(y_train.iloc[val_idx], val_pred_bnn)
        fold_scores_bnn.append(fold_r2_bnn)
        fold_predictions_bnn[val_idx] = val_pred_bnn
        
        print(f"   Fold {fold+1} R²: {fold_r2_bnn:.4f}")
    
    # Calculate overall CV performance
    cv_r2_bnn = r2_score(y_train, fold_predictions_bnn)
    mean_fold_r2_bnn = np.mean(fold_scores_bnn)
    std_fold_r2_bnn = np.std(fold_scores_bnn)
    
    print(f"   📊 CV R²: {cv_r2_bnn:.4f}, Mean Fold R²: {mean_fold_r2_bnn:.4f} ± {std_fold_r2_bnn:.4f}")
    
    # Store results
    result_bnn = {
        'params': params_bnn,
        'cv_r2': cv_r2_bnn,
        'mean_fold_r2': mean_fold_r2_bnn,
        'std_fold_r2': std_fold_r2_bnn,
        'fold_scores': fold_scores_bnn
    }
    cv_results_bnn.append(result_bnn)
    
    # Update best parameters
    if cv_r2_bnn > best_cv_score_bnn:
        best_cv_score_bnn = cv_r2_bnn
        best_params_bnn = params_bnn
        best_cv_predictions_bnn = fold_predictions_bnn.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR BAYESIAN-STYLE NN:")
print(f"Layers: {best_params_bnn['layers']}")
print(f"Dropout Rates: {best_params_bnn['dropout_rates']}")
print(f"Learning Rate: {best_params_bnn['learning_rate']}")
print(f"Best CV R²: {best_cv_score_bnn:.4f}")

# ============================================================
# TRAIN FINAL MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL BAYESIAN-STYLE NN WITH BEST PARAMETERS...")

# Create final model with best parameters
bnn_model = create_bayesian_style_model(
    input_dim=X_train_s.shape[1],
    layers=best_params_bnn['layers'],
    dropout_rates=best_params_bnn['dropout_rates'],
    learning_rate=best_params_bnn['learning_rate']
)

print(f"✅ FINAL BAYESIAN-STYLE NN ARCHITECTURE: 8 → {' → '.join(map(str, best_params_bnn['layers']))} → 1")
bnn_model.summary()

# ============================================================
# FINAL TRAINING WITH UNCERTAINTY QUANTIFICATION
# ============================================================
print("\n🎯 FINAL TRAINING WITH MC DROPOUT UNCERTAINTY...")

# Callbacks for final training
early_stopping_bnn_final = EarlyStopping(
    monitor='val_loss',
    patience=150,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_bnn_final = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final model
history_bnn = bnn_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_10_s, y_val_10),
    epochs=1000,
    batch_size=16,
    verbose=1,
    callbacks=[early_stopping_bnn_final, reduce_lr_bnn_final],
    shuffle=True
)

# ============================================================
# MONTE CARLO DROPOUT PREDICTIONS WITH UNCERTAINTY
# ============================================================
print("📊 PERFORMING MONTE CARLO DROPOUT PREDICTIONS WITH UNCERTAINTY ESTIMATION...")

def predict_mc_dropout(model, X, n_samples=100):
    """
    Monte Carlo Dropout sampling for uncertainty estimation
    """
    preds = np.array([model(X, training=True).numpy().flatten() for _ in range(n_samples)])
    return preds.mean(axis=0), preds.std(axis=0)

# Predictions with uncertainty using MC Dropout
y_pred_log_train_bnn_mean, y_pred_log_train_bnn_std = predict_mc_dropout(bnn_model, X_train_s)
y_pred_log_val_10_bnn_mean, y_pred_log_val_10_bnn_std = predict_mc_dropout(bnn_model, X_val_10_s)
y_pred_log_test_10_bnn_mean, y_pred_log_test_10_bnn_std = predict_mc_dropout(bnn_model, X_test_10_s)

# ============================================================
# METRICS FUNCTION FOR BAYESIAN-STYLE NN
# ============================================================
def compute_metrics_bnn(y_true, y_pred):
    r2_bnn = r2_score(y_true, y_pred)
    mse_bnn = mean_squared_error(y_true, y_pred)
    mae_bnn = mean_absolute_error(y_true, y_pred)
    return r2_bnn, mse_bnn, mae_bnn

# Calculate metrics
r2_bnn_train, mse_bnn_train, mae_bnn_train = compute_metrics_bnn(y_train, y_pred_log_train_bnn_mean)
r2_bnn_val_10, mse_bnn_val_10, mae_bnn_val_10 = compute_metrics_bnn(y_val_10, y_pred_log_val_10_bnn_mean)
r2_bnn_test_10, mse_bnn_test_10, mae_bnn_test_10 = compute_metrics_bnn(y_test_10, y_pred_log_test_10_bnn_mean)

# ============================================================
# COMPREHENSIVE RESULTS
# ============================================================
print("\n📊 BAYESIAN-STYLE NEURAL NETWORK PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_bnn_train:.4f}")
print(f"Validation R² (log): {r2_bnn_val_10:.4f}")
print(f"Testing R² (log):    {r2_bnn_test_10:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_bnn:.4f}")

print(f"\nCross-Validation Fold R² Scores:")
for i, result_bnn in enumerate(cv_results_bnn):
    if result_bnn['params'] == best_params_bnn:
        print(f"  🏆 Best Model Fold R²: {[f'{score:.4f}' for score in result_bnn['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_bnn_train:.4f}")
print(f"Validation MSE (log): {mse_bnn_val_10:.4f}")
print(f"Testing MSE (log):   {mse_bnn_test_10:.4f}")

print(f"\nTraining MAE (log):   {mae_bnn_train:.4f}")
print(f"Validation MAE (log): {mae_bnn_val_10:.4f}")
print(f"Testing MAE (log):    {mae_bnn_test_10:.4f}")

# Uncertainty statistics
print(f"\n📊 UNCERTAINTY QUANTIFICATION (MC Dropout):")
print(f"Training Uncertainty (std): {np.mean(y_pred_log_train_bnn_std):.4f}")
print(f"Testing Uncertainty (std):  {np.mean(y_pred_log_test_10_bnn_std):.4f}")

print(f"\nR² Difference (Train - Test): {r2_bnn_train - r2_bnn_test_10:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_bnn_train - r2_bnn_test_10) > 0.2 else "✅ GOOD")

# ============================================================
# HYPERPARAMETER COMPARISON
# ============================================================
print("\n🔍 BAYESIAN-STYLE NN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_bnn in enumerate(cv_results_bnn):
    marker = "🏆" if result_bnn['params'] == best_params_bnn else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_bnn['cv_r2']:.4f}, "
          f"Layers = {result_bnn['params']['layers']}, "
          f"Dropout = {result_bnn['params']['dropout_rates']}")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR BNN
# ============================================================
y_pred_train_bnn_actual = np.exp(y_pred_log_train_bnn_mean)
y_train_actual = np.exp(y_train)
y_pred_test_10_bnn_actual = np.exp(y_pred_log_test_10_bnn_mean)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions with uncertainty
print("\nFirst 10 Predictions - Test Set (LOG SCALE) with Uncertainty")
comparison_log_df_bnn = pd.DataFrame({
    "Actual (log)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_bnn_mean[:10],
    "Uncertainty (±)": y_pred_log_test_10_bnn_std[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_bnn_mean[:10]
}).round(4)
print(comparison_log_df_bnn)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_bnn = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_bnn_actual[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_bnn_actual[:10]
}).round(2)
print(comparison_actual_df_bnn)

# ============================================================
# UNCERTAINTY ANALYSIS
# ============================================================
print(f"\n🔍 UNCERTAINTY ANALYSIS (MC Dropout):")
print(f"Average Prediction Uncertainty: {np.mean(y_pred_log_test_10_bnn_std):.4f}")
print(f"Max Prediction Uncertainty: {np.max(y_pred_log_test_10_bnn_std):.4f}")
print(f"Min Prediction Uncertainty: {np.min(y_pred_log_test_10_bnn_std):.4f}")

# ============================================================
# TRAINING HISTORY VISUALIZATION
# ============================================================
print("\n📈 PLOTTING TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_bnn.history['loss'], label='Training Loss')
plt.plot(history_bnn.history['val_loss'], label='Validation Loss')
plt.title('Bayesian-Style NN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_bnn.history['mae'], label='Training MAE')
plt.plot(history_bnn.history['val_mae'], label='Validation MAE')
plt.title('Bayesian-Style NN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# UNCERTAINTY VISUALIZATION
# ============================================================
print("\n📊 PLOTTING PREDICTION UNCERTAINTIES...")
plt.figure(figsize=(10, 6))

# Sort by actual values for better visualization
sort_idx = np.argsort(y_test_10)
y_test_sorted = y_test_10[sort_idx]
y_pred_sorted = y_pred_log_test_10_bnn_mean[sort_idx]
y_std_sorted = y_pred_log_test_10_bnn_std[sort_idx]

plt.errorbar(range(len(y_test_sorted)), y_pred_sorted, yerr=y_std_sorted, 
             fmt='o', alpha=0.7, label='Predictions ±1σ', capsize=3)
plt.plot(range(len(y_test_sorted)), y_test_sorted, 'r-', label='Actual Values', linewidth=2)
plt.xlabel('Test Samples (Sorted)')
plt.ylabel('Log Rupture Time')
plt.title('Bayesian-Style NN: Predictions with Uncertainty Bounds')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ BAYESIAN-STYLE NEURAL NETWORK WITH UNCERTAINTY QUANTIFICATION COMPLETED!")
print("="*60)

### BNN Plots 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_bnn = y_train - y_pred_log_train_bnn_mean
test_residuals_bnn = y_test_10 - y_pred_log_test_10_bnn_mean

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_bnn_mean, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_bnn_mean, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("BNN: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_bnn, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_bnn, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("BNN: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_bnn, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_bnn, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("BNN: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 Scatter Band Analysis: BNN

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: BNN
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR BNN...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_bnn = y_pred_test_10_bnn_actual / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_bnn = np.mean((scatter_ratio_bnn >= 0.5) & (scatter_ratio_bnn <= 2.0)) * 100
within_1_5x_bnn = np.mean((scatter_ratio_bnn >= 0.67) & (scatter_ratio_bnn <= 1.5)) * 100
within_3x_bnn = np.mean((scatter_ratio_bnn >= 0.33) & (scatter_ratio_bnn <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_bnn = np.mean(scatter_ratio_bnn < 1.0) * 100  # Under-predictions
optimistic_bnn = np.mean(scatter_ratio_bnn > 1.0) * 100    # Over-predictions

print(f"📊 BNN SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_bnn:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_bnn:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_bnn:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_bnn:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_bnn:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_bnn_actual, alpha=0.7, color='blue', s=50, 
           label=f'BNN Predictions ({within_2x_bnn:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_bnn_actual.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_bnn_actual.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('BNN: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_bnn, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_bnn)
mean_ratio = np.mean(scatter_ratio_bnn)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'BNN: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_bnn:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_bnn)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_bnn):.2f}
Within ±2: {within_2x_bnn:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### One by one predictions

In [ ]:
# ============================================================
# 📊 PREDICTIONS FOR ANALYSIS
# ============================================================
print("📊 EXTRACTING PREDICTION VALUES...")

# Get predictions in actual scale
y_pred_test_10_actual = np.exp(y_pred_log_test_10_bnn_mean)
y_test_10_actual = np.exp(y_test_10)

# Calculate percentage errors
percentage_errors = ((y_test_10_actual - y_pred_test_10_actual) / y_test_10_actual) * 100

# Create simple comparison table
comparison_df = pd.DataFrame({
    'Actual': y_test_10_actual,
    'Predicted': y_pred_test_10_actual,
    'Error_%': percentage_errors
}).round(2)

print("PREDICTIONS VS ACTUAL VALUES:")
print(comparison_df)

### 1D CONVOLUTIONAL NEURAL NETWORK (CNN)

In [ ]:
# ============================================================
# 🔍 1D CONVOLUTIONAL NEURAL NETWORK (CNN)
# ============================================================
print("🔍 IMPLEMENTING 1D CONVOLUTIONAL NEURAL NETWORK (CNN)...")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE 1D CNN ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_cnn_model(input_dim, filters=[64, 32], kernel_sizes=[3, 3], dense_units=[32, 16], 
                    dropout_rate=0.2, l2_reg=0.001, learning_rate=0.001):
    """
    Create flexible 1D CNN architecture for hyperparameter tuning
    """
    model_cnn = Sequential()
    
    # Input layer - reshape for 1D convolution
    model_cnn.add(tf.keras.layers.InputLayer(input_shape=(input_dim, 1)))
    
    # Convolutional layers
    for i, (filt, kernel) in enumerate(zip(filters, kernel_sizes)):
        model_cnn.add(Conv1D(filters=filt, kernel_size=kernel, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=l2(l2_reg),
                           padding='same'))
        model_cnn.add(BatchNormalization())
        model_cnn.add(MaxPooling1D(pool_size=2))
        model_cnn.add(Dropout(dropout_rate))
    
    # Flatten and dense layers
    model_cnn.add(Flatten())
    
    for units in dense_units:
        model_cnn.add(Dense(units, activation='relu',
                          kernel_initializer='he_normal',
                          kernel_regularizer=l2(l2_reg)))
        model_cnn.add(BatchNormalization())
        model_cnn.add(Dropout(dropout_rate * 0.8))
    
    # Output layer
    model_cnn.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_cnn = Adam(learning_rate=learning_rate)
    model_cnn.compile(optimizer=optimizer_cnn, loss='mse', metrics=['mae'])
    
    return model_cnn

# ============================================================
# RESHAPE DATA FOR 1D CNN
# ============================================================
print("🔄 Reshaping data for 1D CNN...")
X_train_cnn = X_train_s.reshape(X_train_s.shape[0], X_train_s.shape[1], 1)
X_val_cnn = X_val_10_s.reshape(X_val_10_s.shape[0], X_val_10_s.shape[1], 1)
X_test_cnn = X_test_10_s.reshape(X_test_10_s.shape[0], X_test_10_s.shape[1], 1)

print(f"✅ CNN Data Shapes:")
print(f"X_train_cnn: {X_train_cnn.shape}")
print(f"X_val_cnn: {X_val_cnn.shape}")
print(f"X_test_cnn: {X_test_cnn.shape}")

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR CNN...")

# Define hyperparameter combinations for CNN
hyperparam_combinations_cnn = [
    # Combination 1: Higher capacity (tests upper limit)
    {'filters': [64, 32], 'kernel_sizes': [3, 3], 'dense_units': [32, 16], 
     'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
    
    # Combination 2: Your proven [32, 24] pattern (most promising)
    {'filters': [32, 24], 'kernel_sizes': [5, 3], 'dense_units': [24, 12], 
     'dropout_rate': 0.3, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    
    # Combination 3: 3-layer CNN (tests depth)
    {'filters': [32, 24, 16], 'kernel_sizes': [3, 3, 2], 'dense_units': [16, 8], 
     'dropout_rate': 0.4, 'l2_reg': 0.005, 'learning_rate': 0.0001},
    
    # Combination 4: Minimalist 2-layer
    {'filters': [24, 16], 'kernel_sizes': [3, 3], 'dense_units': [32], 
     'dropout_rate': 0.2, 'l2_reg': 0.001, 'learning_rate': 0.001},
]

# K-Fold setup
kf_cnn = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_cnn = None
best_cv_score_cnn = -float('inf')
cv_results_cnn = []

print(f"🔄 Testing {len(hyperparam_combinations_cnn)} CNN architecture combinations...")

for i, params_cnn in enumerate(hyperparam_combinations_cnn):
    print(f"\n🧩 Testing CNN Combination {i+1}: {params_cnn}")
    
    fold_scores_cnn = []
    fold_predictions_cnn = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_cnn.split(X_train_cnn)):
        # Create model with current hyperparameters
        model_cnn_cv = create_cnn_model(
            input_dim=X_train_cnn.shape[1],
            filters=params_cnn['filters'],
            kernel_sizes=params_cnn['kernel_sizes'],
            dense_units=params_cnn['dense_units'],
            dropout_rate=params_cnn['dropout_rate'],
            l2_reg=params_cnn['l2_reg'],
            learning_rate=params_cnn['learning_rate']
        )
        
        # Callbacks
        early_stopping_cnn = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_cnn = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_cnn_cv = model_cnn_cv.fit(
            X_train_cnn[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_cnn[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_cnn, reduce_lr_cnn]
        )
        
        # Get predictions and score
        val_pred_cnn = model_cnn_cv.predict(X_train_cnn[val_idx], verbose=0).flatten()
        fold_r2_cnn = r2_score(y_train.iloc[val_idx], val_pred_cnn)
        fold_scores_cnn.append(fold_r2_cnn)
        fold_predictions_cnn[val_idx] = val_pred_cnn
        
        print(f"   Fold {fold+1} R²: {fold_r2_cnn:.4f}")
    
    # Calculate overall CV performance
    cv_r2_cnn = r2_score(y_train, fold_predictions_cnn)
    mean_fold_r2_cnn = np.mean(fold_scores_cnn)
    std_fold_r2_cnn = np.std(fold_scores_cnn)
    
    print(f"   📊 CV R²: {cv_r2_cnn:.4f}, Mean Fold R²: {mean_fold_r2_cnn:.4f} ± {std_fold_r2_cnn:.4f}")
    
    # Store results
    result_cnn = {
        'params': params_cnn,
        'cv_r2': cv_r2_cnn,
        'mean_fold_r2': mean_fold_r2_cnn,
        'std_fold_r2': std_fold_r2_cnn,
        'fold_scores': fold_scores_cnn
    }
    cv_results_cnn.append(result_cnn)
    
    # Update best parameters
    if cv_r2_cnn > best_cv_score_cnn:
        best_cv_score_cnn = cv_r2_cnn
        best_params_cnn = params_cnn
        best_cv_predictions_cnn = fold_predictions_cnn.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR CNN:")
print(f"Filters: {best_params_cnn['filters']}")
print(f"Kernel Sizes: {best_params_cnn['kernel_sizes']}")
print(f"Dense Units: {best_params_cnn['dense_units']}")
print(f"Dropout: {best_params_cnn['dropout_rate']}")
print(f"L2 Reg: {best_params_cnn['l2_reg']}")
print(f"Learning Rate: {best_params_cnn['learning_rate']}")
print(f"Best CV R²: {best_cv_score_cnn:.4f}")

# ============================================================
# TRAIN FINAL CNN MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL CNN WITH BEST PARAMETERS...")

# Create final model with best parameters
cnn_model = create_cnn_model(
    input_dim=X_train_cnn.shape[1],
    filters=best_params_cnn['filters'],
    kernel_sizes=best_params_cnn['kernel_sizes'],
    dense_units=best_params_cnn['dense_units'],
    dropout_rate=best_params_cnn['dropout_rate'],
    l2_reg=best_params_cnn['l2_reg'],
    learning_rate=best_params_cnn['learning_rate']
)

print(f"✅ FINAL CNN ARCHITECTURE:")
print(f"Input: {X_train_cnn.shape[1]} features → Conv1D {best_params_cnn['filters']} → Dense {best_params_cnn['dense_units']} → Output")
cnn_model.summary()

# Final training callbacks for CNN
early_stopping_cnn = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_cnn = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final CNN model on full training data with validation set
history_cnn = cnn_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_cnn, reduce_lr_cnn],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR CNN
# ============================================================
print("📊 EVALUATING FINAL CNN MODEL...")

# Predictions
y_pred_log_train_cnn = cnn_model.predict(X_train_cnn).flatten()
y_pred_log_val_10_cnn = cnn_model.predict(X_val_cnn).flatten()
y_pred_log_test_10_cnn = cnn_model.predict(X_test_cnn).flatten()

# Metrics function for CNN
def compute_metrics_cnn(y_true, y_pred):
    r2_cnn = r2_score(y_true, y_pred)
    mse_cnn = mean_squared_error(y_true, y_pred)
    mae_cnn = mean_absolute_error(y_true, y_pred)
    return r2_cnn, mse_cnn, mae_cnn

# Calculate metrics for CNN
r2_cnn_train, mse_cnn_train, mae_cnn_train = compute_metrics_cnn(y_train, y_pred_log_train_cnn)
r2_cnn_val_10, mse_cnn_val_10, mae_cnn_val_10 = compute_metrics_cnn(y_val_10, y_pred_log_val_10_cnn)
r2_cnn_test_10, mse_cnn_test_10, mae_cnn_test_10 = compute_metrics_cnn(y_test_10, y_pred_log_test_10_cnn)

# ============================================================
# COMPREHENSIVE RESULTS FOR CNN
# ============================================================
print("\n📊 1D CNN PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_cnn_train:.4f}")
print(f"Validation R² (log): {r2_cnn_val_10:.4f}")
print(f"Testing R² (log):    {r2_cnn_test_10:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_cnn:.4f}")

print(f"\nCross-Validation Fold R² Scores for CNN:")
for i, result_cnn in enumerate(cv_results_cnn):
    if result_cnn['params'] == best_params_cnn:
        print(f"  🏆 Best CNN Model Fold R²: {[f'{score:.4f}' for score in result_cnn['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_cnn_train:.4f}")
print(f"Validation MSE (log): {mse_cnn_val_10:.4f}")
print(f"Testing MSE (log):   {mse_cnn_test_10:.4f}")

print(f"\nTraining MAE (log):   {mae_cnn_train:.4f}")
print(f"Validation MAE (log): {mae_cnn_val_10:.4f}")
print(f"Testing MAE (log):    {mae_cnn_test_10:.4f}")

print(f"\nR² Difference (Train - Test): {r2_cnn_train - r2_cnn_test_10:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_cnn_train - r2_cnn_test_10) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR CNN
# ============================================================
y_pred_train_cnn = np.exp(y_pred_log_train_cnn)
y_train_actual = np.exp(y_train)
y_pred_test_10_cnn = np.exp(y_pred_log_test_10_cnn)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_cnn = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_cnn[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_cnn[:10]
}).round(4)
print(comparison_log_df_cnn)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_cnn = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_cnn[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_cnn[:10]
}).round(2)
print(comparison_actual_df_cnn)

# ============================================================
# HYPERPARAMETER COMPARISON FOR CNN
# ============================================================
print("\n🔍 CNN HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_cnn in enumerate(cv_results_cnn):
    marker = "🏆" if result_cnn['params'] == best_params_cnn else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_cnn['cv_r2']:.4f}, "
          f"Filters = {result_cnn['params']['filters']}, "
          f"Kernels = {result_cnn['params']['kernel_sizes']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR CNN
# ============================================================
print("\n📈 PLOTTING CNN TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['loss'], label='Training Loss')
plt.plot(history_cnn.history['val_loss'], label='Validation Loss')
plt.title('1D CNN: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['mae'], label='Training MAE')
plt.plot(history_cnn.history['val_mae'], label='Validation MAE')
plt.title('1D CNN: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ 1D CONVOLUTIONAL NEURAL NETWORK WITH K-FOLD CV COMPLETED!")
print("="*60)

### 1D CNN Plots 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_cnn = y_train - y_pred_log_train_cnn
test_residuals_cnn = y_test_10 - y_pred_log_test_10_cnn

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_cnn, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_cnn, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("CNN: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_cnn, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_cnn, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("CNN: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_cnn, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_cnn, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("CNN: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 SCATTER BAND ANALYSIS: 1D CNN

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: CNN
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR CNN...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_cnn = y_pred_test_10_cnn / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_cnn = np.mean((scatter_ratio_cnn >= 0.5) & (scatter_ratio_cnn <= 2.0)) * 100
within_1_5x_cnn = np.mean((scatter_ratio_cnn >= 0.67) & (scatter_ratio_cnn <= 1.5)) * 100
within_3x_cnn = np.mean((scatter_ratio_cnn >= 0.33) & (scatter_ratio_cnn <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_cnn = np.mean(scatter_ratio_cnn < 1.0) * 100  # Under-predictions
optimistic_cnn = np.mean(scatter_ratio_cnn > 1.0) * 100    # Over-predictions

print(f"📊 CNN SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_cnn:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_cnn:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_cnn:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_cnn:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_cnn:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_cnn, alpha=0.7, color='blue', s=50, 
           label=f'CNN Predictions ({within_2x_cnn:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_cnn.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_cnn.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('CNN: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_cnn, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_cnn)
mean_ratio = np.mean(scatter_ratio_cnn)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'CNN: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_cnn:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_cnn)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_cnn):.2f}
Within ±2: {within_2x_cnn:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### One by one predictions

In [ ]:
# ============================================================
# 📊 PREDICTIONS FOR ANALYSIS
# ============================================================
print("📊 EXTRACTING PREDICTION VALUES...")

# Get predictions in actual scale
y_pred_test_10_actual = np.exp(y_pred_log_test_10_cnn)
y_test_10_actual = np.exp(y_test_10)

# Calculate percentage errors
percentage_errors = ((y_test_10_actual - y_pred_test_10_actual) / y_test_10_actual) * 100

# Create simple comparison table
comparison_df = pd.DataFrame({
    'Actual': y_test_10_actual,
    'Predicted': y_pred_test_10_actual,
    'Error_%': percentage_errors
}).round(2)

print("PREDICTIONS VS ACTUAL VALUES:")
print(comparison_df)

### LONG SHORT-TERM MEMORY (LSTM) NETWORK

In [ ]:
# ============================================================
# 🔄 LONG SHORT-TERM MEMORY (LSTM) NETWORK
# ============================================================
print("🔄 IMPLEMENTING LONG SHORT-TERM MEMORY (LSTM) NETWORK...")
print("="*60) 

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split

# ============================================================
# FLEXIBLE LSTM ARCHITECTURE FOR HYPERPARAMETER TUNING
# ============================================================
def create_lstm_model(input_dim, lstm_units=[64, 32], dense_units=[32, 16], 
                     dropout_rate=0.2, recurrent_dropout=0.1, l2_reg=0.001, 
                     learning_rate=0.001, return_sequences=True):
    """
    Create flexible LSTM architecture for hyperparameter tuning
    """
    model_lstm = Sequential()
    
    # Input layer - reshape for LSTM (samples, timesteps, features)
    model_lstm.add(tf.keras.layers.InputLayer(input_shape=(input_dim, 1)))
    
    # LSTM layers
    for i, units in enumerate(lstm_units):
        return_seq = (i < len(lstm_units) - 1)  # Return sequences for all but last layer
        model_lstm.add(LSTM(units=units, 
                          activation='tanh',
                          recurrent_activation='sigmoid',
                          kernel_initializer='glorot_uniform',
                          recurrent_initializer='orthogonal',
                          kernel_regularizer=l2(l2_reg),
                          recurrent_regularizer=l2(l2_reg),
                          dropout=dropout_rate,
                          recurrent_dropout=recurrent_dropout,
                          return_sequences=return_seq))
        model_lstm.add(BatchNormalization())
    
    # Dense layers
    for units in dense_units:
        model_lstm.add(Dense(units, activation='relu',
                           kernel_initializer='he_normal',
                           kernel_regularizer=l2(l2_reg)))
        model_lstm.add(BatchNormalization())
        model_lstm.add(Dropout(dropout_rate * 0.8))
    
    # Output layer
    model_lstm.add(Dense(1, activation='linear'))
    
    # Compile
    optimizer_lstm = Adam(learning_rate=learning_rate)
    model_lstm.compile(optimizer=optimizer_lstm, loss='mse', metrics=['mae'])
    
    return model_lstm

# ============================================================
# RESHAPE DATA FOR LSTM
# ============================================================
print("🔄 Reshaping data for LSTM...")
# Treat each feature as a time step (8 timesteps, 1 feature per timestep)
X_train_lstm = X_train_s.reshape(X_train_s.shape[0], X_train_s.shape[1], 1)
X_val_lstm = X_val_10_s.reshape(X_val_10_s.shape[0], X_val_10_s.shape[1], 1)
X_test_lstm = X_test_10_s.reshape(X_test_10_s.shape[0], X_test_10_s.shape[1], 1)

print(f"✅ LSTM Data Shapes:")
print(f"X_train_lstm: {X_train_lstm.shape}")  # (samples, 8 timesteps, 1 feature)
print(f"X_val_lstm: {X_val_lstm.shape}")
print(f"X_test_lstm: {X_test_lstm.shape}")

# ============================================================
# HYPERPARAMETER TUNING WITH K-FOLD CROSS VALIDATION
# ============================================================
print("🔍 PERFORMING HYPERPARAMETER TUNING WITH 5-FOLD CV FOR LSTM...")

# Define hyperparameter combinations for LSTM
hyperparam_combinations_lstm = [
    # Combination 1: Moderate 2-layer LSTM
    {'lstm_units': [64, 32], 'dense_units': [32, 16], 'dropout_rate': 0.2, 
     'recurrent_dropout': 0.1, 'l2_reg': 0.001, 'learning_rate': 0.001},
    
    # Combination 2: Your proven [32, 24] pattern adapted for LSTM
    {'lstm_units': [32, 24], 'dense_units': [24, 12], 'dropout_rate': 0.3, 
     'recurrent_dropout': 0.2, 'l2_reg': 0.002, 'learning_rate': 0.0005},
    
    # Combination 3: Lightweight 2-layer LSTM
    {'lstm_units': [32, 16], 'dense_units': [16, 8], 'dropout_rate': 0.4, 
     'recurrent_dropout': 0.3, 'l2_reg': 0.005, 'learning_rate': 0.0001},
    
    # Combination 4: Single LSTM layer (simplest)
    {'lstm_units': [64], 'dense_units': [32, 16], 'dropout_rate': 0.2, 
     'recurrent_dropout': 0.1, 'l2_reg': 0.001, 'learning_rate': 0.001},
]

# K-Fold setup
kf_lstm = KFold(n_splits=5, shuffle=True, random_state=42)
best_params_lstm = None
best_cv_score_lstm = -float('inf')
cv_results_lstm = []

print(f"🔄 Testing {len(hyperparam_combinations_lstm)} LSTM architecture combinations...")

for i, params_lstm in enumerate(hyperparam_combinations_lstm):
    print(f"\n🧩 Testing LSTM Combination {i+1}: {params_lstm}")
    
    fold_scores_lstm = []
    fold_predictions_lstm = np.zeros(len(y_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf_lstm.split(X_train_lstm)):
        # Create model with current hyperparameters
        model_lstm_cv = create_lstm_model(
            input_dim=X_train_lstm.shape[1],
            lstm_units=params_lstm['lstm_units'],
            dense_units=params_lstm['dense_units'],
            dropout_rate=params_lstm['dropout_rate'],
            recurrent_dropout=params_lstm['recurrent_dropout'],
            l2_reg=params_lstm['l2_reg'],
            learning_rate=params_lstm['learning_rate']
        )
        
        # Callbacks
        early_stopping_lstm = EarlyStopping(
            monitor='val_loss', patience=80, restore_best_weights=True, verbose=0
        )
        reduce_lr_lstm = ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=40, verbose=0
        )
        
        # Train model
        history_lstm_cv = model_lstm_cv.fit(
            X_train_lstm[train_idx], y_train.iloc[train_idx],
            validation_data=(X_train_lstm[val_idx], y_train.iloc[val_idx]),
            epochs=400,
            batch_size=8,
            verbose=0,
            callbacks=[early_stopping_lstm, reduce_lr_lstm]
        )
        
        # Get predictions and score
        val_pred_lstm = model_lstm_cv.predict(X_train_lstm[val_idx], verbose=0).flatten()
        fold_r2_lstm = r2_score(y_train.iloc[val_idx], val_pred_lstm)
        fold_scores_lstm.append(fold_r2_lstm)
        fold_predictions_lstm[val_idx] = val_pred_lstm
        
        print(f"   Fold {fold+1} R²: {fold_r2_lstm:.4f}")
    
    # Calculate overall CV performance
    cv_r2_lstm = r2_score(y_train, fold_predictions_lstm)
    mean_fold_r2_lstm = np.mean(fold_scores_lstm)
    std_fold_r2_lstm = np.std(fold_scores_lstm)
    
    print(f"   📊 CV R²: {cv_r2_lstm:.4f}, Mean Fold R²: {mean_fold_r2_lstm:.4f} ± {std_fold_r2_lstm:.4f}")
    
    # Store results
    result_lstm = {
        'params': params_lstm,
        'cv_r2': cv_r2_lstm,
        'mean_fold_r2': mean_fold_r2_lstm,
        'std_fold_r2': std_fold_r2_lstm,
        'fold_scores': fold_scores_lstm
    }
    cv_results_lstm.append(result_lstm)
    
    # Update best parameters
    if cv_r2_lstm > best_cv_score_lstm:
        best_cv_score_lstm = cv_r2_lstm
        best_params_lstm = params_lstm
        best_cv_predictions_lstm = fold_predictions_lstm.copy()

# Display best parameters
print(f"\n🎯 BEST HYPERPARAMETERS FOR LSTM:")
print(f"LSTM Units: {best_params_lstm['lstm_units']}")
print(f"Dense Units: {best_params_lstm['dense_units']}")
print(f"Dropout: {best_params_lstm['dropout_rate']}")
print(f"Recurrent Dropout: {best_params_lstm['recurrent_dropout']}")
print(f"L2 Reg: {best_params_lstm['l2_reg']}")
print(f"Learning Rate: {best_params_lstm['learning_rate']}")
print(f"Best CV R²: {best_cv_score_lstm:.4f}")

# ============================================================
# TRAIN FINAL LSTM MODEL WITH BEST PARAMETERS
# ============================================================
print("\n🔄 TRAINING FINAL LSTM WITH BEST PARAMETERS...")

# Create final model with best parameters
lstm_model = create_lstm_model(
    input_dim=X_train_lstm.shape[1],
    lstm_units=best_params_lstm['lstm_units'],
    dense_units=best_params_lstm['dense_units'],
    dropout_rate=best_params_lstm['dropout_rate'],
    recurrent_dropout=best_params_lstm['recurrent_dropout'],
    l2_reg=best_params_lstm['l2_reg'],
    learning_rate=best_params_lstm['learning_rate']
)

print(f"✅ FINAL LSTM ARCHITECTURE:")
print(f"Input: {X_train_lstm.shape[1]} timesteps → LSTM {best_params_lstm['lstm_units']} → Dense {best_params_lstm['dense_units']} → Output")
lstm_model.summary()

# Final training callbacks for LSTM
early_stopping_lstm = EarlyStopping(
    monitor='val_loss',
    patience=100,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0001
)

reduce_lr_lstm = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=50,
    min_lr=1e-7,
    verbose=1
)

# Train final LSTM model on full training data with validation set
history_lstm = lstm_model.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val_10),
    epochs=1000,
    batch_size=8,
    verbose=1,
    callbacks=[early_stopping_lstm, reduce_lr_lstm],
    shuffle=True
)

# ============================================================
# PREDICTIONS AND METRICS FOR LSTM
# ============================================================
print("📊 EVALUATING FINAL LSTM MODEL...")

# Predictions
y_pred_log_train_lstm = lstm_model.predict(X_train_lstm).flatten()
y_pred_log_val_10_lstm = lstm_model.predict(X_val_lstm).flatten()
y_pred_log_test_10_lstm = lstm_model.predict(X_test_lstm).flatten()

# Metrics function for LSTM
def compute_metrics_lstm(y_true, y_pred):
    r2_lstm = r2_score(y_true, y_pred)
    mse_lstm = mean_squared_error(y_true, y_pred)
    mae_lstm = mean_absolute_error(y_true, y_pred)
    return r2_lstm, mse_lstm, mae_lstm

# Calculate metrics for LSTM
r2_lstm_train, mse_lstm_train, mae_lstm_train = compute_metrics_lstm(y_train, y_pred_log_train_lstm)
r2_lstm_val_10, mse_lstm_val_10, mae_lstm_val_10 = compute_metrics_lstm(y_val_10, y_pred_log_val_10_lstm)
r2_lstm_test_10, mse_lstm_test_10, mae_lstm_test_10 = compute_metrics_lstm(y_test_10, y_pred_log_test_10_lstm)

# ============================================================
# COMPREHENSIVE RESULTS FOR LSTM
# ============================================================
print("\n📊 LSTM PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log):   {r2_lstm_train:.4f}")
print(f"Validation R² (log): {r2_lstm_val_10:.4f}")
print(f"Testing R² (log):    {r2_lstm_test_10:.4f}")
print(f"Cross-Validation R² (log): {best_cv_score_lstm:.4f}")

print(f"\nCross-Validation Fold R² Scores for LSTM:")
for i, result_lstm in enumerate(cv_results_lstm):
    if result_lstm['params'] == best_params_lstm:
        print(f"  🏆 Best LSTM Model Fold R²: {[f'{score:.4f}' for score in result_lstm['fold_scores']]}")

print(f"\nTraining MSE (log):  {mse_lstm_train:.4f}")
print(f"Validation MSE (log): {mse_lstm_val_10:.4f}")
print(f"Testing MSE (log):   {mse_lstm_test_10:.4f}")

print(f"\nTraining MAE (log):   {mae_lstm_train:.4f}")
print(f"Validation MAE (log): {mae_lstm_val_10:.4f}")
print(f"Testing MAE (log):    {mae_lstm_test_10:.4f}")

print(f"\nR² Difference (Train - Test): {r2_lstm_train - r2_lstm_test_10:.4f}")
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_lstm_train - r2_lstm_test_10) > 0.2 else "✅ GOOD")

# ============================================================
# PREDICTIONS IN BOTH SCALES FOR LSTM
# ============================================================
y_pred_train_lstm = np.exp(y_pred_log_train_lstm)
y_train_actual = np.exp(y_train)
y_pred_test_10_lstm = np.exp(y_pred_log_test_10_lstm)
y_test_10_actual = np.exp(y_test_10)

# First 10 predictions
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_lstm = pd.DataFrame({
    "Actual (log Rupture Time)": y_test_10[:10],
    "Predicted (log)": y_pred_log_test_10_lstm[:10],
    "Error": y_test_10[:10] - y_pred_log_test_10_lstm[:10]
}).round(4)
print(comparison_log_df_lstm)

print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_lstm = pd.DataFrame({
    "Actual (Rupture Time)": y_test_10_actual[:10],
    "Predicted": y_pred_test_10_lstm[:10],
    "Error": y_test_10_actual[:10] - y_pred_test_10_lstm[:10]
}).round(2)
print(comparison_actual_df_lstm)

# ============================================================
# HYPERPARAMETER COMPARISON FOR LSTM
# ============================================================
print("\n🔍 LSTM HYPERPARAMETER COMPARISON RESULTS:")
print("="*50)
for i, result_lstm in enumerate(cv_results_lstm):
    marker = "🏆" if result_lstm['params'] == best_params_lstm else "  "
    print(f"{marker} Comb {i+1}: CV R² = {result_lstm['cv_r2']:.4f}, "
          f"LSTM Units = {result_lstm['params']['lstm_units']}, "
          f"Dense Units = {result_lstm['params']['dense_units']}")

# ============================================================
# TRAINING HISTORY VISUALIZATION FOR LSTM
# ============================================================
print("\n📈 PLOTTING LSTM TRAINING HISTORY...")
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.title('LSTM: Loss Evolution')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history_lstm.history['mae'], label='Training MAE')
plt.plot(history_lstm.history['val_mae'], label='Validation MAE')
plt.title('LSTM: MAE Evolution')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ LONG SHORT-TERM MEMORY NETWORK WITH K-FOLD CV COMPLETED!")
print("="*60)

### LSTM Plots 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Residuals
# -----------------------------
train_residuals_lstm = y_train - y_pred_log_train_lstm
test_residuals_lstm = y_test_10 - y_pred_log_test_10_lstm

# ============================================================
# 1️⃣ Predicted vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=y_pred_log_train_lstm, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_lstm, label='Test', color='orange', s=60)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'k--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Predicted log(Rupture Time)")
plt.title("LSTM: Predicted vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ Residuals vs Actual (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_train, y=train_residuals_lstm, label='Train', color='steelblue', s=60)
sns.scatterplot(x=y_test_10, y=test_residuals_lstm, label='Test', color='orange', s=60)
plt.axhline(0, color='k', linestyle='--', alpha=0.7)
plt.xlabel("Actual log(Rupture Time)")
plt.ylabel("Residuals")
plt.title("LSTM: Residuals vs Actual (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ Histogram of Residuals (Log Scale)
# ============================================================
plt.figure(figsize=(7,6))
sns.histplot(train_residuals_lstm, color='steelblue', label='Train', kde=True, bins=30)
sns.histplot(test_residuals_lstm, color='orange', label='Test', kde=True, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Count") 
plt.title("LSTM: Histogram of Residuals (Log Scale)")
plt.legend()
plt.tight_layout()
plt.show()


### ±2 SCATTER BAND ANALYSIS: LSTM

In [ ]:
# ============================================================
# 🔬 ±2 SCATTER BAND ANALYSIS: LSTM
# ============================================================
print("🔬 ANALYZING ±2 SCATTER BAND FOR LSTM...")
print("="*60)

# Calculate scatter ratios (Predicted/Actual)
scatter_ratio_lstm = y_pred_test_10_lstm / y_test_10_actual

# Scatter band metrics (FIXED variable names - no ± symbol)
within_2x_lstm = np.mean((scatter_ratio_lstm >= 0.5) & (scatter_ratio_lstm <= 2.0)) * 100
within_1_5x_lstm = np.mean((scatter_ratio_lstm >= 0.67) & (scatter_ratio_lstm <= 1.5)) * 100
within_3x_lstm = np.mean((scatter_ratio_lstm >= 0.33) & (scatter_ratio_lstm <= 3.0)) * 100

# Conservative vs Optimistic predictions
conservative_lstm = np.mean(scatter_ratio_lstm < 1.0) * 100  # Under-predictions
optimistic_lstm = np.mean(scatter_ratio_lstm > 1.0) * 100    # Over-predictions

print(f"📊 LSTM SCATTER BAND ANALYSIS:")
print(f"   ±2 Scatter Band:  {within_2x_lstm:.1f}% of predictions")
print(f"   ±1.5 Scatter Band: {within_1_5x_lstm:.1f}% of predictions") 
print(f"   ±3 Scatter Band:  {within_3x_lstm:.1f}% of predictions")
print(f"   Conservative predictions (<1.0): {conservative_lstm:.1f}%")
print(f"   Optimistic predictions (>1.0): {optimistic_lstm:.1f}%")

# ============================================================
# SCATTER BAND PLOT
# ============================================================
plt.figure(figsize=(10, 8))

# Scatter plot
plt.scatter(y_test_10_actual, y_pred_test_10_lstm, alpha=0.7, color='blue', s=50, 
           label=f'LSTM Predictions ({within_2x_lstm:.1f}% within ±2 band)')

# Reference lines
max_val = max(y_test_10_actual.max(), y_pred_test_10_lstm.max())
min_val = min(y_test_10_actual.min(), y_pred_test_10_lstm.min())

# Perfect prediction line
plt.plot([min_val, max_val], [min_val, max_val], 'k-', label='Perfect Prediction', linewidth=2)

# ±2 scatter band lines
plt.plot([min_val, max_val], [0.5*min_val, 0.5*max_val], 'r--', label='±2 Scatter Band', linewidth=1.5)
plt.plot([min_val, max_val], [2*min_val, 2*max_val], 'r--', linewidth=1.5)

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Actual Rupture Time (hours)')
plt.ylabel('Predicted Rupture Time (hours)')
plt.title('LSTM: ±2 Scatter Band Analysis\n(Actual Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# SCATTER RATIO DISTRIBUTION (IMPROVED VERSION)
# ============================================================
plt.figure(figsize=(10, 6))

# Use consistent binning and add statistics
n, bins, patches = plt.hist(scatter_ratio_lstm, bins=50, alpha=0.7, color='blue', 
                           edgecolor='black', density=False)

plt.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Perfect Prediction (1.0)')
plt.axvline(0.5, color='orange', linestyle='--', linewidth=1.5, label='±2 Band Limits')
plt.axvline(2.0, color='orange', linestyle='--', linewidth=1.5)

# Add statistical annotations
median_ratio = np.median(scatter_ratio_lstm)
mean_ratio = np.mean(scatter_ratio_lstm)

plt.axvline(median_ratio, color='green', linestyle=':', linewidth=2, 
           label=f'Median Ratio: {median_ratio:.2f}')

plt.xlabel('Predicted / Actual Ratio')
plt.ylabel('Frequency')
plt.title(f'LSTM: Scatter Ratio Distribution\n'
          f'Mean: {mean_ratio:.2f}, Median: {median_ratio:.2f}, '
          f'Within ±2: {within_2x_lstm:.1f}%')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add text box with key statistics
textstr = f'''Key Statistics:
N = {len(scatter_ratio_lstm)}
Mean = {mean_ratio:.2f}
Median = {median_ratio:.2f}
Std = {np.std(scatter_ratio_lstm):.2f}
Within ±2: {within_2x_lstm:.1f}%'''
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.95, 0.95, textstr, transform=plt.gca().transAxes, 
               verticalalignment='top', horizontalalignment='right', bbox=props)

plt.show()

### One by one predictions

In [ ]:
# ============================================================
# 📊 PREDICTIONS FOR ANALYSIS
# ============================================================
print("📊 EXTRACTING PREDICTION VALUES...")

# Get predictions in actual scale
y_pred_test_10_actual = np.exp(y_pred_log_test_10_cnn)
y_test_10_actual = np.exp(y_test_10)

# Calculate percentage errors
percentage_errors = ((y_test_10_actual - y_pred_test_10_actual) / y_test_10_actual) * 100

# Create simple comparison table
comparison_df = pd.DataFrame({
    'Actual': y_test_10_actual,
    'Predicted': y_pred_test_10_actual,
    'Error_%': percentage_errors
}).round(2)

print("PREDICTIONS VS ACTUAL VALUES:")
print(comparison_df)

### DL Complete Grid

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# -----------------------------
# 📊 3×2 GRID WITH ALL PLOTS
# -----------------------------
sns.set_style("white")
plt.rcParams.update({
    'figure.figsize': (10, 15),  # Width, Height for 2×3 grid
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'lines.linewidth': 2,
    'lines.markersize': 4,
    'font.family': 'Arial',
}) 

fig = plt.figure()
gs = gridspec.GridSpec(3, 2, figure=fig)  # 3 rows, 2 columns

# Plot 1-5: Your existing plots
ax1 = fig.add_subplot(gs[0, 0])  # Row 0, Col 0
sns.scatterplot(x=y_train, y=y_pred_log_train_dnn_3layer, label='Train',
                color='#1f77b4', s=40, alpha=0.9, edgecolor='none')
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_dnn_3layer, label='Test',
                color='#ff7f0e', s=40, alpha=0.9, edgecolor='none')

# Perfect prediction line
lims = [min(y_train.min(), y_test_10.min()), max(y_train.max(), y_test_10.max())]
padding = 0.02 * (lims[1] - lims[0])  # 2% padding for visual margin
lims = [lims[0] - padding, lims[1] + padding]
plt.plot(lims, lims, 'k--', lw=1.5, alpha=0.8, zorder=3)

# Axis labels and formatting
plt.xlabel("Actual log(Creep Life)")
plt.ylabel("Predicted log(Creep Life)")
plt.title("(a) DNN", loc='center', fontweight='bold')

# Apply limits and ticks
plt.xlim(lims)
plt.ylim(lims)
plt.tick_params(direction='out', length=4, width=1,
                bottom=True, left=True, top=False, right=False)  # ticks all sides

# Legend and layout
plt.legend(frameon=False)

ax2 = fig.add_subplot(gs[0, 1])  # Row 0, Col 1  
sns.scatterplot(x=y_train, y=y_pred_log_train_bnn_mean, label='Train',
                color='#1f77b4', s=40, alpha=0.9, edgecolor='none')
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_bnn_mean, label='Test',
                color='#ff7f0e', s=40, alpha=0.9, edgecolor='none')

# Perfect prediction line
lims = [min(y_train.min(), y_test_10.min()), max(y_train.max(), y_test_10.max())]
padding = 0.02 * (lims[1] - lims[0])  # 2% padding for visual margin
lims = [lims[0] - padding, lims[1] + padding]
plt.plot(lims, lims, 'k--', lw=1.5, alpha=0.8, zorder=3)

# Axis labels and formatting
plt.xlabel("Actual log(Creep Life)")
plt.ylabel("Predicted log(Creep Life)")
plt.title("(b) BNN", loc='center', fontweight='bold')

# Apply limits and ticks
plt.xlim(lims)
plt.ylim(lims)
plt.tick_params(direction='out', length=4, width=1,
                bottom=True, left=True, top=False, right=False)  # ticks all sides

# Legend and layout
plt.legend(frameon=False)

ax3 = fig.add_subplot(gs[1, 0])  # Row 1, Col 0
sns.scatterplot(x=y_train, y=y_pred_log_train_cnn, label='Train',
                color='#1f77b4', s=40, alpha=0.9, edgecolor='none')
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_cnn, label='Test',
                color='#ff7f0e', s=40, alpha=0.9, edgecolor='none')

# Perfect prediction line
lims = [min(y_train.min(), y_test_10.min()), max(y_train.max(), y_test_10.max())]
padding = 0.02 * (lims[1] - lims[0])  # 2% padding for visual margin
lims = [lims[0] - padding, lims[1] + padding]
plt.plot(lims, lims, 'k--', lw=1.5, alpha=0.8, zorder=3)

# Axis labels and formatting
plt.xlabel("Actual log(Creep Life)")
plt.ylabel("Predicted log(Creep Life)")
plt.title("(c) CNN", loc='center', fontweight='bold')

# Apply limits and ticks
plt.xlim(lims)
plt.ylim(lims)
plt.tick_params(direction='out', length=4, width=1,
                bottom=True, left=True, top=False, right=False)  # ticks all sides

# Legend and layout
plt.legend(frameon=False)

ax4 = fig.add_subplot(gs[1, 1])  # Row 1, Col 1
sns.scatterplot(x=y_train, y=y_pred_log_train_lstm, label='Train',
                color='#1f77b4', s=40, alpha=0.9, edgecolor='none')
sns.scatterplot(x=y_test_10, y=y_pred_log_test_10_lstm, label='Test',
                color='#ff7f0e', s=40, alpha=0.9, edgecolor='none')

# Perfect prediction line
lims = [min(y_train.min(), y_test_10.min()), max(y_train.max(), y_test_10.max())]
padding = 0.02 * (lims[1] - lims[0])  # 2% padding for visual margin
lims = [lims[0] - padding, lims[1] + padding]
plt.plot(lims, lims, 'k--', lw=1.5, alpha=0.8, zorder=3)

# Axis labels and formatting
plt.xlabel("Actual log(Creep Life)")
plt.ylabel("Predicted log(Creep Life)")
plt.title("(d) LSTM", loc='center', fontweight='bold')

# Apply limits and ticks
plt.xlim(lims)
plt.ylim(lims)
plt.tick_params(direction='out', length=4, width=1,
                bottom=True, left=True, top=False, right=False)  # ticks all sides

# Legend and layout
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig('complete_grid.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()

### Comprehensive Table

In [ ]:
import time
from time import perf_counter
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import r2_score

# Import your model creation functions
# (Make sure these are defined or imported before running)
# from your_model_file import create_dnn_3layer_model, create_bayesian_style_model, create_cnn_model, create_lstm_model

# -------------------------
# CONFIG
# -------------------------
REPEATS_TRAIN = 3
REPEATS_INF = 50
WARMUP_INF = 5

# -------------------------
# MODEL CONFIGURATIONS (WITH YOUR ACTUAL BEST HYPERPARAMETERS)
# -------------------------
INPUT_DIM = X_train_s.shape[1]

MODEL_CONFIGS = {
    'DNN': {
        'function': create_dnn_3layer_model,
        'kwargs': best_params_dnn_3layer,  # Use your actual best params
        'epochs': 1000,
        'batch_size': 8,
        'patience': 100,
        'data_reshape': None  # No reshaping needed
    },
    'BNN': {
        'function': create_bayesian_style_model,
        'kwargs': best_params_bnn,  # Use your actual best params
        'epochs': 1000,
        'batch_size': 16,
        'patience': 150,
        'data_reshape': None
    },
    'CNN': {
        'function': create_cnn_model,
        'kwargs': best_params_cnn,  # Use your actual best params
        'epochs': 1000,
        'batch_size': 8,
        'patience': 100,
        'data_reshape': 'cnn'  # Needs reshaping
    },
    'LSTM': {
        'function': create_lstm_model,
        'kwargs': best_params_lstm,  # Use your actual best params
        'epochs': 1000,
        'batch_size': 8,
        'patience': 100,
        'data_reshape': 'lstm'  # Needs reshaping
    }
}

# -------------------------
# DATA PREPARATION
# -------------------------
def prepare_data(X_train, X_test, reshape_type=None):
    """Reshape data based on model requirements"""
    if reshape_type == 'cnn' or reshape_type == 'lstm':
        X_train_reshaped = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        return X_train_reshaped, X_test_reshaped
    return X_train, X_test

# -------------------------
# DL PARAMETER COUNTING
# -------------------------
def get_dl_param_count(model):
    """Count trainable parameters for DL models"""
    try:
        return int(model.count_params())
    except:
        return np.nan

# -------------------------
# DL TRAINING TIME WITH EARLY STOPPING
# -------------------------
def measure_dl_train_time(model_fn, model_kwargs, X_train, y_train, X_val, y_val,
                          epochs, batch_size, patience, repeats=3):
    """
    Measure training time with early stopping (matches actual training procedure)
    """
    times = []
    
    for i in range(repeats):
        print(f"    Training iteration {i+1}/{repeats}...")
        
        # Create fresh model
        fresh_model = model_fn(**model_kwargs)
        
        # Callbacks (same as your actual training)
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=patience,
            restore_best_weights=True,
            verbose=0,
            min_delta=0.0001
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=patience//2,
            min_lr=1e-7,
            verbose=0
        )
        
        # Train and time
        t0 = perf_counter()
        history = fresh_model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping, reduce_lr],
            verbose=0,
            shuffle=True
        )
        t1 = perf_counter()
        
        actual_epochs = len(history.history['loss'])
        times.append(t1 - t0)
        print(f"    Training time: {t1-t0:.2f}s (stopped at epoch {actual_epochs})")
        
        # Cleanup
        del fresh_model
        tf.keras.backend.clear_session()
    
    return float(np.mean(times)), float(np.std(times))

# -------------------------
# DL INFERENCE TIME
# -------------------------
def measure_dl_inference_time(fitted_model, X, repeats=50, warmup=5):
    """Measure inference time per sample"""
    batch_size = min(2000, len(X))
    if batch_size < 100:
        batch_size = len(X)
    
    test_batch = X[:batch_size]
    
    # Warmup
    for _ in range(warmup):
        _ = fitted_model.predict(test_batch, verbose=0)
    
    times = []
    for _ in range(repeats):
        t0 = perf_counter()
        _ = fitted_model.predict(test_batch, verbose=0)
        t1 = perf_counter()
        times.append(t1 - t0)
    
    times = np.array(times)
    mean_per_sample = times.mean() / batch_size
    std_per_sample = times.std() / batch_size
    return float(mean_per_sample), float(std_per_sample)

# -------------------------
# EVALUATE DL MODELS
# -------------------------
rows = []

for name, config in MODEL_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating DL Model: {name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train_model, X_test_model = prepare_data(
        X_train_s, X_test_10_s, config['data_reshape']
    )
    _, X_val_model = prepare_data(
        X_train_s, X_val_10_s, config['data_reshape']
    )
    
    # Add input_dim to kwargs
    model_kwargs = config['kwargs'].copy()
    model_kwargs['input_dim'] = INPUT_DIM
    
    # Create model template
    try:
        model_template = config['function'](**model_kwargs)
        print(f"  Model created successfully")
    except Exception as e:
        print(f"  Model creation failed: {e}")
        continue
    
    # 1) Parameter count
    try:
        pcount = get_dl_param_count(model_template)
        print(f"  Parameters: {pcount:,}")
    except Exception as e:
        pcount = np.nan
        print(f"  Parameter count failed: {e}")
    
    # 2) Training time (with early stopping)
    print(f"  Training with early stopping (max {config['epochs']} epochs)...")
    try:
        train_time_mean, train_time_std = measure_dl_train_time(
            config['function'], model_kwargs,
            X_train_model, y_train,
            X_val_model, y_val_10,
            epochs=config['epochs'],
            batch_size=config['batch_size'],
            patience=config['patience'],
            repeats=REPEATS_TRAIN
        )
        print(f"  Training time: {train_time_mean:.2f}s ± {train_time_std:.2f}s")
    except Exception as e:
        print(f"  Training timing failed: {e}")
        train_time_mean, train_time_std = np.nan, np.nan
    
    # 3) Create fitted model for inference
    try:
        print("  Creating fitted model for inference...")
        fitted_model = config['function'](**model_kwargs)
        
        early_stopping = EarlyStopping(
            monitor='val_loss', patience=config['patience'],
            restore_best_weights=True, verbose=0
        )
        
        fitted_model.fit(
            X_train_model, y_train,
            validation_data=(X_val_model, y_val_10),
            epochs=config['epochs'],
            batch_size=config['batch_size'],
            callbacks=[early_stopping],
            verbose=0
        )
    except Exception as e:
        print(f"  Creating fitted model failed: {e}")
        fitted_model = model_template
    
    # 4) Inference time
    try:
        print("  Measuring inference time...")
        inf_mean, inf_std = measure_dl_inference_time(
            fitted_model, X_test_model, REPEATS_INF, WARMUP_INF
        )
        print(f"  Inference time: {inf_mean * 1000:.4f}ms per sample")
    except Exception as e:
        print(f"  Inference timing failed: {e}")
        inf_mean, inf_std = np.nan, np.nan
    
    # 5) Test metrics
    try:
        print("  Calculating metrics...")
        y_pred_log = fitted_model.predict(X_test_model, verbose=0).flatten()
        rmse_log = np.sqrt(np.mean((y_test_10 - y_pred_log)**2))
        r2_log = r2_score(y_test_10, y_pred_log)
        print(f"  Test RMSE (log): {rmse_log:.4f}")
        print(f"  Test R² (log): {r2_log:.4f}")
    except Exception as e:
        print(f"  Metrics failed: {e}")
        rmse_log, r2_log = np.nan, np.nan
    
    # Store results
    rows.append({
        'Model': name,
        'ParamCount': pcount,
        'TrainTime_s_mean': train_time_mean,
        'TrainTime_s_std': train_time_std,
        'InfTime_ms_per_sample': inf_mean * 1000,
        'InfTime_ms_per_sample_std': inf_std * 1000,
        'TestRMSE_log': rmse_log,
        'TestR2_log': r2_log
    })
    
    # Cleanup
    del fitted_model, model_template
    tf.keras.backend.clear_session()

# -------------------------
# BUILD DATAFRAME
# -------------------------
df_dl = pd.DataFrame(rows)
df_dl = df_dl.sort_values('ParamCount', na_position='last').reset_index(drop=True)

def safe_relcol(series):
    finite = series.replace([np.inf, -np.inf], np.nan).dropna()
    if finite.empty:
        return np.nan
    fastest = finite.min()
    return series / fastest

df_dl['TrainTime_rel'] = safe_relcol(df_dl['TrainTime_s_mean'])
df_dl['InfTime_rel'] = safe_relcol(df_dl['InfTime_ms_per_sample'])
df_dl['ParamCount_log10'] = df_dl['ParamCount'].apply(
    lambda x: np.log10(x) if pd.notnull(x) and x>0 else np.nan
)

# Print and save
pd.options.display.float_format = '{:,.4f}'.format
print("\n" + "="*60)
print("DL Models Performance vs Cost Table")
print("="*60)
print(df_dl)

df_dl.to_excel('dl_performance_vs_cost_table.xlsx', index=False)
print(f"\n✅ Saved to 'dl_performance_vs_cost_table.xlsx'")

# Combine with ML
try:
    df_ml = pd.read_excel('performance_vs_cost_table.xlsx')
    df_combined = pd.concat([df_ml, df_dl], ignore_index=True)
    df_combined.to_excel('ALL_models_performance_vs_cost.xlsx', index=False)
    print("✅ Combined saved to 'ALL_models_performance_vs_cost.xlsx'")
except:
    print("ℹ️  ML file not found")

In [ ]:
# For DL TEST set (46 points)
log_rupture_time_dl_test = y_test_10.values 

range1_dl = np.sum((log_rupture_time_dl_test >= 0.25) & (log_rupture_time_dl_test < 4.0))
range2_dl = np.sum((log_rupture_time_dl_test >= 4.0) & (log_rupture_time_dl_test < 8.0))
range3_dl = np.sum((log_rupture_time_dl_test >= 8.0) & (log_rupture_time_dl_test < 11.0))
range4_dl = np.sum((log_rupture_time_dl_test >= 11.0) & (log_rupture_time_dl_test <= 13.0))

print("DL TEST SET (46 points):")
print(f"Range 1 (0.25-4.0): {range1_dl} points")
print(f"Range 2 (4.0-8.0): {range2_dl} points")
print(f"Range 3 (8.0-11.0): {range3_dl} points")
print(f"Range 4 (11.0-13.0): {range4_dl} points")
print(f"Total: {range1_dl+range2_dl+range3_dl+range4_dl}")

### Error Analyses

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ============================================================
# ERROR ANALYSIS BY RUPTURE LIFE RANGE - DL MODELS
# ============================================================

# Your actual DL test values (46 points)
y_test_log_dl = y_test_10

# Get predictions from all 4 DL models (using your variable names)
predictions_dl = {
    'DNN': y_pred_log_test_10_dnn_3layer,
    'BNN': y_pred_log_test_10_bnn_mean,
    'CNN': y_pred_log_test_10_cnn,
    'LSTM': y_pred_log_test_10_lstm
}

# Define ranges (same as ML)
ranges = [
    (0.25, 4.0, "Range 1 (0.25-4.0)", "Very short-term"),
    (4.0, 8.0, "Range 2 (4.0-8.0)", "Short-to-medium"),
    (8.0, 11.0, "Range 3 (8.0-11.0)", "Long-term"),
    (11.0, 13.0, "Range 4 (11.0-13.0)", "Very long-term")
]

# Store results
all_results_dl = []

# Calculate metrics for each model and each range
for model_name, y_pred_log in predictions_dl.items():
    for range_min, range_max, range_label, range_desc in ranges:
        # Get indices in this range
        mask = (y_test_log_dl >= range_min) & (y_test_log_dl < range_max)
        
        y_true_range = y_test_log_dl[mask]
        y_pred_range = y_pred_log[mask]
        
        n_points = len(y_true_range)
        
        if n_points > 0:
            rmse = np.sqrt(mean_squared_error(y_true_range, y_pred_range))
            mae = mean_absolute_error(y_true_range, y_pred_range)
            me = np.mean(y_pred_range - y_true_range)  # Mean Error (bias)
            
            all_results_dl.append({
                'Model': model_name,
                'Range': range_label,
                'Description': range_desc,
                'N': n_points,
                'RMSE': round(rmse, 4),
                'MAE': round(mae, 4),
                'ME': round(me, 4)
            })
        else:
            all_results_dl.append({
                'Model': model_name,
                'Range': range_label,
                'Description': range_desc,
                'N': 0,
                'RMSE': np.nan,
                'MAE': np.nan,
                'ME': np.nan
            })

# Create DataFrame
df_dl_results = pd.DataFrame(all_results_dl)

# Display results
print("\n" + "="*80)
print("📊 ERROR METRICS BY RUPTURE LIFE RANGE - DL MODELS")
print("="*80)
print(df_dl_results.to_string(index=False))

# Save to CSV
df_dl_results.to_csv('dl_error_by_range.csv', index=False)
print("\n✅ Results saved to 'dl_error_by_range.csv'")

# Summary: Show which models perform best in each range
print("\n" + "="*80)
print("🏆 BEST MODEL PER RANGE (Based on RMSE)")
print("="*80)
for range_label in df_dl_results['Range'].unique():
    range_data = df_dl_results[df_dl_results['Range'] == range_label]
    best_model = range_data.loc[range_data['RMSE'].idxmin()]
    print(f"{range_label}: {best_model['Model']} (RMSE: {best_model['RMSE']:.4f})")

print("\n" + "="*80)
print("⚠️ MODELS WITH HIGHEST BIAS (Mean Error) IN EACH RANGE")
print("="*80)
for range_label in df_dl_results['Range'].unique():
    range_data = df_dl_results[df_dl_results['Range'] == range_label]
    # Highest absolute mean error
    worst_bias = range_data.loc[range_data['ME'].abs().idxmax()]
    bias_direction = "Over-predicting" if worst_bias['ME'] > 0 else "Under-predicting"
    print(f"{range_label}: {worst_bias['Model']} - {bias_direction} (ME: {worst_bias['ME']:.4f})")

### ML Codes (Just Necessary Here)

In [ ]:
#Data loading for ML
test_data = pd.read_excel("2.25Cr1Mo_Test_SelectedFeatures.xlsx")
X_test = test_data[features] 
y_test = test_data['Lg(CL)'] 
X_test_s = s.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 LINEAR REGRESSION PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 TRAINING LINEAR REGRESSION ON LOG TARGET (Evaluating in LOG SCALE)...")

pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------
pipeline_lr.fit(X_train, y_train)

y_pred_log_train_lr = pipeline_lr.predict(X_train)
y_pred_log_test_lr = pipeline_lr.predict(X_test)

# ------------------------------------------------------------
# METRICS FUNCTION (LOG SCALE)
# ------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    r2_lr = r2_score(y_true, y_pred)
    mse_lr = mean_squared_error(y_true, y_pred)
    mae_lr = mean_absolute_error(y_true, y_pred)
    return r2_lr, mse_lr, mae_lr

r2_lr_train, mse_lr_train, mae_lr_train = compute_metrics(y_train, y_pred_log_train_lr)
r2_lr_test, mse_lr_test, mae_lr_test = compute_metrics(y_test, y_pred_log_test_lr)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_lr = cross_val_predict(pipeline_lr, X_train, y_train, cv=kf)
cv_r2_lr = r2_score(y_train, y_cv_pred_log_lr)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------
print("\n📊 LINEAR REGRESSION PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_lr_train:.4f}")
print(f"Testing  R² (log): {r2_lr_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_lr:.4f}\n")

print(f"Training MSE (log): {mse_lr_train:.4f}")
print(f"Testing  MSE (log): {mse_lr_test:.4f}")
print(f"Training MAE (log): {mae_lr_train:.4f}")
print(f"Testing  MAE (log): {mae_lr_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_lr_train - r2_lr_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_lr_train - r2_lr_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# SHOW FIRST 10 PREDICTIONS (LOG SCALE and Actual Scale)
# ------------------------------------------------------------

y_pred_train_lr = np.exp(y_pred_log_train_lr)
y_pred_test_lr = np.exp(y_pred_log_test_lr)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# Log-scale:

print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_lr = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_lr[:10],
    "Error": y_test[:10] - y_pred_log_test_lr[:10]
}).round(4)
print(comparison_log_df_lr)

# Actual scale: 

print("\nFirst 10 Predictions - Test Set (Actual Scale)")
comparison_actual_df_lr = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_lr[:10],
    "Error": y_test_actual[:10] - y_pred_test_lr[:10]
}).round(2)
print(comparison_actual_df_lr)

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 SVR PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 PERFORMING SVR HYPERPARAMETER TUNING ON LOG TARGET...")

# Pipeline for scaling + SVR
pipeline_svr = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR())
])

# Parameter grid
param_grid_svr = {
    'svr__C': [0.1, 1, 10, 100],
    'svr__gamma': ['scale', 'auto', 0.1, 0.01],
    'svr__kernel': ['rbf', 'linear']
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_svr = GridSearchCV(
    pipeline_svr,
    param_grid_svr,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------
grid_search_svr.fit(X_train, y_train)
best_svr_pipeline = grid_search_svr.best_estimator_

print(f"🎯 Best SVR Parameters: {grid_search_svr.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_svr.best_score_:.4f}")

# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_svr = pd.DataFrame(grid_search_svr.cv_results_)
print(f"\n🔍 Top 5 SVR parameter combinations:")
print(results_df_svr[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ------------------------------------------------------------
# PREDICTIONS (LOG SCALE)
# ------------------------------------------------------------
y_pred_log_train_svr = best_svr_pipeline.predict(X_train)
y_pred_log_test_svr = best_svr_pipeline.predict(X_test)

# ------------------------------------------------------------
# METRICS FUNCTION (LOG SCALE)
# ------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    r2_svr = r2_score(y_true, y_pred)
    mse_svr = mean_squared_error(y_true, y_pred)
    mae_svr = mean_absolute_error(y_true, y_pred)
    return r2_svr, mse_svr, mae_svr

# Training & Testing metrics (log scale)
r2_svr_train, mse_svr_train, mae_svr_train = compute_metrics(y_train, y_pred_log_train_svr)
r2_svr_test, mse_svr_test, mae_svr_test = compute_metrics(y_test, y_pred_log_test_svr)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_svr = cross_val_predict(best_svr_pipeline, X_train, y_train, cv=kf)
cv_r2_svr = r2_score(y_train, y_cv_pred_log_svr)

# ------------------------------------------------------------
# DISPLAY RESULTS (LOG SCALE)
# ------------------------------------------------------------
print("\n📊 SVR PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_svr_train:.4f}")
print(f"Testing  R² (log): {r2_svr_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_svr:.4f}\n")

print(f"Training MSE (log): {mse_svr_train:.4f}")
print(f"Testing  MSE (log): {mse_svr_test:.4f}")
print(f"Training MAE (log): {mae_svr_train:.4f}")
print(f"Testing  MAE (log): {mae_svr_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_svr_train - r2_svr_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_svr_train - r2_svr_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE
# ------------------------------------------------------------
y_pred_train_svr = np.exp(y_pred_log_train_svr)
y_pred_test_svr = np.exp(y_pred_log_test_svr)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_svr = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_svr[:10],
    "Error": y_test[:10] - y_pred_log_test_svr[:10]
}).round(4)
print(comparison_log_df_svr)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_svr = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_svr[:10],
    "Error": y_test_actual[:10] - y_pred_test_svr[:10]
}).round(2)
print(comparison_actual_df_svr)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 RANDOM FOREST PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 Performing Random Forest hyperparameter tuning on LOG target...")

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    'rf__n_estimators': [100, 200, 500],
    'rf__max_depth': [None, 5, 10, 20],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__max_features': ['sqrt', 'log2', None]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_rf = GridSearchCV(
    pipeline_rf,
    param_grid_rf,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------
grid_search_rf.fit(X_train, y_train)
best_rf_pipeline = grid_search_rf.best_estimator_

print(f"🎯 Best RF Parameters: {grid_search_rf.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_rf.best_score_:.4f}")


# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_rf = pd.DataFrame(grid_search_rf.cv_results_)
print(f"\n🔍 Top 5 RF parameter combinations:")
print(results_df_rf[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ------------------------------------------------------------
# PREDICTIONS (LOG SCALE)
# ------------------------------------------------------------
y_pred_log_train_rf = best_rf_pipeline.predict(X_train)
y_pred_log_test_rf = best_rf_pipeline.predict(X_test)

# ------------------------------------------------------------
# METRICS FUNCTION (LOG SCALE)
# ------------------------------------------------------------
def compute_metrics(y_true, y_pred):
    r2_rf = r2_score(y_true, y_pred)
    mse_rf = mean_squared_error(y_true, y_pred)
    mae_rf = mean_absolute_error(y_true, y_pred)
    return r2_rf, mse_rf, mae_rf

# Training & Testing metrics (log scale)
r2_rf_train, mse_rf_train, mae_rf_train = compute_metrics(y_train, y_pred_log_train_rf)
r2_rf_test, mse_rf_test, mae_rf_test = compute_metrics(y_test, y_pred_log_test_rf)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_rf = cross_val_predict(best_rf_pipeline, X_train, y_train, cv=kf)
cv_r2_rf = r2_score(y_train, y_cv_pred_log_rf)

# ------------------------------------------------------------
# DISPLAY RESULTS (LOG SCALE)
# ------------------------------------------------------------
print("\n📊 RANDOM FOREST PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_rf_train:.4f}")
print(f"Testing  R² (log): {r2_rf_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_rf:.4f}\n")

print(f"Training MSE (log): {mse_rf_train:.4f}")
print(f"Testing  MSE (log): {mse_rf_test:.4f}")
print(f"Training MAE (log): {mae_rf_train:.4f}")
print(f"Testing  MAE (log): {mae_rf_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_rf_train - r2_rf_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_rf_train - r2_rf_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE

y_pred_train_rf = np.exp(y_pred_log_train_rf)
y_train_actual = np.exp(y_train)
y_pred_test_rf = np.exp(y_pred_log_test_rf)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_rf = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_rf[:10],
    "Error": y_test[:10] - y_pred_log_test_rf[:10]
}).round(4)
print(comparison_log_df_rf)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_rf = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_rf[:10],
    "Error": y_test_actual[:10] - y_pred_test_rf[:10]
}).round(2)
print(comparison_actual_df_rf)


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 GRADIENT BOOSTING REGRESSION PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================

print("🔄 Performing Gradient Boosting Regression hyperparameter tuning on LOG target...")

pipeline_gbr = Pipeline([
    ('scaler', StandardScaler()),
    ('gbr', GradientBoostingRegressor(random_state=42))
])

param_grid_gbr = {
    'gbr__n_estimators': [100, 200, 500],
    'gbr__learning_rate': [0.01, 0.05, 0.1],
    'gbr__max_depth': [2, 3, 4],
    'gbr__min_samples_split': [2, 5],
    'gbr__min_samples_leaf': [1, 2]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_gbr = GridSearchCV(
    pipeline_gbr,
    param_grid_gbr,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------

grid_search_gbr.fit(X_train, y_train)
best_gbr_pipeline = grid_search_gbr.best_estimator_

print(f"🎯 Best GBR Parameters: {grid_search_gbr.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_gbr.best_score_:.4f}")

# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_gbr = pd.DataFrame(grid_search_gbr.cv_results_)
print(f"\n🔍 Top 5 GBR parameter combinations:")
print(results_df_gbr[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ============================================================
# PREDICTIONS (LOG SCALE)
# ============================================================
y_pred_log_train_gbr = best_gbr_pipeline.predict(X_train)
y_pred_log_test_gbr = best_gbr_pipeline.predict(X_test)

# ============================================================
# METRICS FUNCTION (LOG SCALE)
# ============================================================

def compute_metrics(y_true, y_pred):
    r2_gbr = r2_score(y_true, y_pred)
    mse_gbr = mean_squared_error(y_true, y_pred)
    mae_gbr = mean_absolute_error(y_true, y_pred)
    return r2_gbr, mse_gbr, mae_gbr

# Training & Testing metrics (log scale)
r2_gbr_train, mse_gbr_train, mae_gbr_train = compute_metrics(y_train, y_pred_log_train_gbr)
r2_gbr_test, mse_gbr_test, mae_gbr_test = compute_metrics(y_test, y_pred_log_test_gbr)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_gbr = cross_val_predict(best_gbr_pipeline, X_train, y_train, cv=kf)
cv_r2_gbr = r2_score(y_train, y_cv_pred_log_gbr)

# ============================================================
# DISPLAY RESULTS (LOG SCALE)
# ============================================================
print("\n📊 GRADIENT BOOSTING REGRESSOR PERFORMANCE (Log Scale)")
print("="*60)
print(f"Training R² (log): {r2_gbr_train:.4f}")
print(f"Testing  R² (log): {r2_gbr_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_gbr:.4f}\n")

print(f"Training MSE (log): {mse_gbr_train:.4f}")
print(f"Testing  MSE (log): {mse_gbr_test:.4f}")
print(f"Training MAE (log): {mae_gbr_train:.4f}")
print(f"Testing  MAE (log): {mae_gbr_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_gbr_train - r2_gbr_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_gbr_train - r2_gbr_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE

y_pred_train_gbr = np.exp(y_pred_log_train_gbr)
y_pred_test_gbr = np.exp(y_pred_log_test_gbr)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_gbr = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_gbr[:10],
    "Error": y_test[:10] - y_pred_log_test_gbr[:10]
}).round(4)
print(comparison_log_df_gbr)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_gbr = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_gbr[:10],
    "Error": y_test_actual[:10] - y_pred_test_gbr[:10]
}).round(2)
print(comparison_actual_df_gbr)


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler  
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

# ============================================================
# 🔄 XGBOOST PIPELINE (LOG TARGET, LOG-SCALE METRICS)
# ============================================================
print("🔄 Performing XGBoost hyperparameter tuning on LOG target...")

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    ))
])

param_grid_xgb = {
    'xgb__n_estimators': [100, 200, 300],
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__subsample': [0.8, 0.9, 1.0],
    'xgb__colsample_bytree': [0.8, 1.0]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search_xgb = GridSearchCV(
    pipeline_xgb,
    param_grid_xgb,
    cv=kf,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    refit=True
)

# ------------------------------------------------------------
# FIT GRID SEARCH
# ------------------------------------------------------------
grid_search_xgb.fit(X_train, y_train)
best_xgb_pipeline = grid_search_xgb.best_estimator_

print(f"🎯 Best XGBoost Parameters: {grid_search_xgb.best_params_}")
print(f"🎯 Best CV R² (log scale): {grid_search_xgb.best_score_:.4f}")

# SHOW TOP PARAMETER COMBINATIONS (This is optional) 
# ------------------------------------------------------------
results_df_xgb = pd.DataFrame(grid_search_xgb.cv_results_)
print(f"\n🔍 Top 5 XGBoost parameter combinations:")
print(results_df_xgb[['params', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(5))
# ------------------------------------------------------------

# ============================================================
# PREDICTIONS (LOG SCALE)
# ============================================================
y_pred_log_train_xgb = best_xgb_pipeline.predict(X_train)
y_pred_log_test_xgb = best_xgb_pipeline.predict(X_test)

# ============================================================
# METRICS FUNCTION (LOG SCALE)
# ============================================================
def compute_metrics(y_true, y_pred):
    r2_xgb = r2_score(y_true, y_pred)
    mse_xgb = mean_squared_error(y_true, y_pred)
    mae_xgb = mean_absolute_error(y_true, y_pred)
    return r2_xgb, mse_xgb, mae_xgb

# Training & Testing metrics (log scale)
r2_xgb_train, mse_xgb_train, mae_xgb_train = compute_metrics(y_train, y_pred_log_train_xgb)
r2_xgb_test, mse_xgb_test, mae_xgb_test = compute_metrics(y_test, y_pred_log_test_xgb)

# ------------------------------------------------------------
# CROSS-VALIDATION R² (LOG SCALE)
# ------------------------------------------------------------
y_cv_pred_log_xgb = cross_val_predict(best_xgb_pipeline, X_train, y_train, cv=kf)
cv_r2_xgb = r2_score(y_train, y_cv_pred_log_xgb)

# ============================================================
# DISPLAY RESULTS (LOG SCALE)
# ============================================================
print("\n📊 XGBOOST PERFORMANCE (LOG SCALE)")
print("="*60)
print(f"Training R² (log): {r2_xgb_train:.4f}")
print(f"Testing  R² (log): {r2_xgb_test:.4f}")
print(f"Cross-Validation R² (log): {cv_r2_xgb:.4f}\n")

print(f"Training MSE (log): {mse_xgb_train:.4f}")
print(f"Testing  MSE (log): {mse_xgb_test:.4f}")
print(f"Training MAE (log): {mae_xgb_train:.4f}")
print(f"Testing  MAE (log): {mae_xgb_test:.4f}")

print("\nR² Difference (Train - Test):", round(r2_xgb_train - r2_xgb_test, 4))
print("Overfitting Indicator:", "⚠️ HIGH (Overfitting)" if (r2_xgb_train - r2_xgb_test) > 0.2 else "✅ GOOD")

# ------------------------------------------------------------
# PREDICTIONS IN BOTH LOG AND ACTUAL SCALE
# ------------------------------------------------------------
y_pred_train_xgb = np.exp(y_pred_log_train_xgb)
y_pred_test_xgb = np.exp(y_pred_log_test_xgb)
y_train_actual = np.exp(y_train)
y_test_actual = np.exp(y_test)

# First 10 predictions in LOG SCALE
print("\nFirst 10 Predictions - Test Set (LOG SCALE)")
comparison_log_df_xgb = pd.DataFrame({
    "Actual (log Rupture Time)": y_test[:10],
    "Predicted (log)": y_pred_log_test_xgb[:10],
    "Error": y_test[:10] - y_pred_log_test_xgb[:10]
}).round(4)
print(comparison_log_df_xgb)

# First 10 predictions in ACTUAL SCALE
print("\nFirst 10 Predictions - Test Set (ACTUAL SCALE)")
comparison_actual_df_xgb = pd.DataFrame({
    "Actual (Rupture Time)": y_test_actual[:10],
    "Predicted": y_pred_test_xgb[:10],
    "Error": y_test_actual[:10] - y_pred_test_xgb[:10]
}).round(2)
print(comparison_actual_df_xgb)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ============================================================
# ERROR ANALYSIS BY RUPTURE LIFE RANGE - ML MODELS
# ============================================================

# Your actual test values
y_test_log = y_test  # This is 'Lg(CL)' - 92 points

# Get predictions from all 5 ML models (using your variable names)
predictions_ml = {
    'LR': y_pred_log_test_lr,      # Linear Regression
    'SVR': y_pred_log_test_svr,    # SVR
    'RF': y_pred_log_test_rf,      # Random Forest
    'GBR': y_pred_log_test_gbr,    # Gradient Boosting
    'XGB': y_pred_log_test_xgb     # XGBoost
}

# Define ranges
ranges = [
    (0.25, 4.0, "Range 1 (0.25-4.0)", "Very short-term"),
    (4.0, 8.0, "Range 2 (4.0-8.0)", "Short-to-medium"),
    (8.0, 11.0, "Range 3 (8.0-11.0)", "Long-term"),
    (11.0, 13.0, "Range 4 (11.0-13.0)", "Very long-term")
]

# Store results
all_results = []

# Calculate metrics for each model and each range
for model_name, y_pred_log in predictions_ml.items():
    for range_min, range_max, range_label, range_desc in ranges:
        # Get indices in this range
        mask = (y_test_log >= range_min) & (y_test_log < range_max)
        
        y_true_range = y_test_log[mask]
        y_pred_range = y_pred_log[mask]
        
        n_points = len(y_true_range)
        
        if n_points > 0:
            rmse = np.sqrt(mean_squared_error(y_true_range, y_pred_range))
            mae = mean_absolute_error(y_true_range, y_pred_range)
            me = np.mean(y_pred_range - y_true_range)  # Mean Error (bias)
            
            all_results.append({
                'Model': model_name,
                'Range': range_label,
                'Description': range_desc,
                'N': n_points,
                'RMSE': round(rmse, 4),
                'MAE': round(mae, 4),
                'ME': round(me, 4)
            })
        else:
            all_results.append({
                'Model': model_name,
                'Range': range_label,
                'Description': range_desc,
                'N': 0,
                'RMSE': np.nan,
                'MAE': np.nan,
                'ME': np.nan
            })

# Create DataFrame
df_ml_results = pd.DataFrame(all_results)

# Display results
print("\n" + "="*80)
print("📊 ERROR METRICS BY RUPTURE LIFE RANGE - ML MODELS")
print("="*80)
print(df_ml_results.to_string(index=False))

# Save to CSV (optional)
df_ml_results.to_csv('ml_error_by_range.csv', index=False)
print("\n✅ Results saved to 'ml_error_by_range.csv'")

# Summary: Show which models perform best in each range
print("\n" + "="*80)
print("🏆 BEST MODEL PER RANGE (Based on RMSE)")
print("="*80)
for range_label in df_ml_results['Range'].unique():
    range_data = df_ml_results[df_ml_results['Range'] == range_label]
    best_model = range_data.loc[range_data['RMSE'].idxmin()]
    print(f"{range_label}: {best_model['Model']} (RMSE: {best_model['RMSE']:.4f})")

print("\n" + "="*80)
print("⚠️ MODELS WITH HIGHEST BIAS (Mean Error) IN EACH RANGE")
print("="*80)
for range_label in df_ml_results['Range'].unique():
    range_data = df_ml_results[df_ml_results['Range'] == range_label]
    # Highest absolute mean error
    worst_bias = range_data.loc[range_data['ME'].abs().idxmax()]
    bias_direction = "Over-predicting" if worst_bias['ME'] > 0 else "Under-predicting"
    print(f"{range_label}: {worst_bias['Model']} - {bias_direction} (ME: {worst_bias['ME']:.4f})")

### Combine ML and DL (Error Analysis)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ============================================================
# PREPARE COMBINED DATA
# ============================================================

# Combine ML and DL results
df_ml_results['Model_Type'] = 'ML'
df_dl_results['Model_Type'] = 'DL'
df_combined = pd.concat([df_ml_results, df_dl_results], ignore_index=True)

# Create figure with multiple subplots
fig = plt.figure(figsize=(11.6, 10.1))

# ============================================================
# PLOT 1: HEATMAP - RMSE ACROSS MODELS AND RANGES
# ============================================================
ax1 = plt.subplot(2, 2, 1)

# Pivot data for heatmap
heatmap_data = df_combined.pivot(index='Model', columns='Range', values='RMSE')
# Reorder ranges
range_order = ['Range 1 (0.25-4.0)', 'Range 2 (4.0-8.0)', 'Range 3 (8.0-11.0)', 'Range 4 (11.0-13.0)']
heatmap_data = heatmap_data[range_order]

# Create heatmap
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn_r', 
            cbar_kws={'label': 'RMSE'}, linewidths=0.5, ax=ax1,
            vmin=0.3, vmax=1.3)

cbar = ax1.collections[0].colorbar
cbar.set_label('RMSE', fontweight='bold', fontsize=12)

ax1.set_title('(a)', 
              fontsize=13, fontweight='bold', pad=10)
ax1.set_xlabel('Creep Rupture Life Range', fontsize=12, fontweight='bold')
ax1.set_ylabel('Model', fontsize=12, fontweight='bold')
ax1.set_xticklabels(['Range 1\n(Very short)', 'Range 2\n(Short-medium)', 
                     'Range 3\n(Long)', 'Range 4\n(Very long)'], rotation=0)

# ============================================================
# PLOT 2: BAR CHART - MEAN ERROR (BIAS) BY MODEL AND RANGE
# ============================================================
ax2 = plt.subplot(2, 2, 2)

# Prepare data
models_order = ['LR', 'SVR', 'RF', 'GBR', 'XGB', 'DNN', 'BNN', 'CNN', 'LSTM']
df_combined['Model'] = pd.Categorical(df_combined['Model'], categories=models_order, ordered=True)
df_combined_sorted = df_combined.sort_values('Model')

# Create grouped bar chart
x = np.arange(len(models_order))
width = 0.2
ranges_short = ['R1', 'R2', 'R3', 'R4']
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i, (range_label, color) in enumerate(zip(range_order, colors)):
    range_data = df_combined_sorted[df_combined_sorted['Range'] == range_label]
    me_values = range_data['ME'].values
    ax2.bar(x + i*width, me_values, width, label=ranges_short[i], color=color, alpha=0.8)

ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Model', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Error (ME)', fontsize=12, fontweight='bold')
ax2.set_title('(b)', 
              fontsize=13, fontweight='bold', pad=10)
ax2.set_xticks(x + width*1.5)
ax2.set_xticklabels(models_order)
ax2.legend(title='Range', loc='upper right', fontsize=6.8)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(-1.5, 1.5)

# Add annotation
ax2.text(0.02, 0.98, 'Positive: Over-prediction\nNegative: Under-prediction', 
         transform=ax2.transAxes, fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

# ============================================================
# PLOT 3: LINE PLOT - RMSE TREND ACROSS RANGES (BEST MODELS)
# ============================================================
ax3 = plt.subplot(2, 2, 3)

# Select best models from each category
best_models = ['SVR', 'BNN']  # Best ML and Best DL
colors_line = {'SVR': '#2ecc71', 'BNN': '#e74c3c'}
markers = {'SVR': 'o', 'BNN': 's'}

for model in best_models:
    model_data = df_combined[df_combined['Model'] == model].sort_values('Range')
    rmse_values = model_data['RMSE'].values
    ax3.plot(range(4), rmse_values, marker=markers[model], linewidth=2.5, 
             markersize=8, label=model, color=colors_line[model], alpha=0.8)

# Add all other models with thin lines
other_models = [m for m in models_order if m not in best_models]
for model in other_models:
    model_data = df_combined[df_combined['Model'] == model].sort_values('Range')
    rmse_values = model_data['RMSE'].values
    linestyle = '--' if model in ['DNN', 'CNN', 'LSTM'] else '-'
    ax3.plot(range(4), rmse_values, linewidth=1, alpha=0.3, 
             linestyle=linestyle, color='gray')

ax3.set_xlabel('Creep Rupture Life Range', fontsize=12, fontweight='bold')
ax3.set_ylabel('RMSE', fontsize=12, fontweight='bold')
ax3.set_title('(c)', 
              fontsize=13, fontweight='bold', pad=10)
ax3.set_xticks(range(4))
ax3.set_xticklabels(['Range 1\n(Very short)', 'Range 2\n(Short-med)', 
                     'Range 3\n(Long)', 'Range 4\n(Very long)'])
ax3.legend(title='Best Models', loc='upper right', fontsize=9)
ax3.grid(True, alpha=0.3)

# ============================================================
# PLOT 4: BOX PLOT - ERROR DISTRIBUTION COMPARISON
# ============================================================
ax4 = plt.subplot(2, 2, 4)

# Create categories for box plot
ml_models = ['LR', 'SVR', 'RF', 'GBR', 'XGB']
dl_models = ['DNN', 'BNN', 'CNN', 'LSTM']

ml_rmse = df_combined[df_combined['Model'].isin(ml_models)]['RMSE'].values
dl_rmse = df_combined[df_combined['Model'].isin(dl_models)]['RMSE'].values

bp = ax4.boxplot([ml_rmse, dl_rmse], labels=['ML Models', 'DL Models'],
                  patch_artist=True, widths=0.6,
                  boxprops=dict(facecolor='lightblue', alpha=0.7),
                  medianprops=dict(color='red', linewidth=2),
                  whiskerprops=dict(linewidth=1.5),
                  capprops=dict(linewidth=1.5))

# Color the boxes
bp['boxes'][0].set_facecolor('#3498db')
bp['boxes'][1].set_facecolor('#e74c3c')
ax4.set_xlabel('Models', fontsize=12, fontweight='bold')
ax4.set_ylabel('RMSE', fontsize=12, fontweight='bold')
ax4.set_title('(d)', 
              fontsize=13, fontweight='bold', pad=10)
ax4.grid(axis='y', alpha=0.3)

# Add mean markers
means = [ml_rmse.mean(), dl_rmse.mean()]
ax4.plot([1, 2], means, 'D', color='green', markersize=8, 
         label=f'Mean', zorder=3)
ax4.legend(loc='upper right', fontsize=8)

# Add statistics text
ml_mean = ml_rmse.mean()
dl_mean = dl_rmse.mean()
ax4.text(0.02, 0.98, f'ML Mean: {ml_mean:.3f}\nDL Mean: {dl_mean:.3f}', 
         transform=ax4.transAxes, fontsize=9, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('error_distribution_analysis.png', dpi=1200, bbox_inches='tight')
plt.show()

print("\n✅ Figure saved as 'error_distribution_analysis.png'")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ============================================================
# FIGURE 2: DETAILED RANGE-SPECIFIC ANALYSIS (ELSEVIER OPTIMIZED)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 10))  # Reduced from (16, 12)
axes = axes.flatten()

range_order = ['Range 1 (0.25-4.0)', 'Range 2 (4.0-8.0)', 'Range 3 (8.0-11.0)', 'Range 4 (11.0-13.0)']
range_titles = ['Range 1: Very Short-term (0.25-4.0 log hours)', 
                'Range 2: Short-to-Medium term (4.0-8.0 log hours)',
                'Range 3: Long-term (8.0-11.0 log hours)', 
                'Range 4: Very Long-term (11.0-13.0 log hours)']

models_order = ['LR', 'SVR', 'RF', 'GBR', 'XGB', 'DNN', 'BNN', 'CNN', 'LSTM']
ml_models = ['LR', 'SVR', 'RF', 'GBR', 'XGB']
dl_models = ['DNN', 'BNN', 'CNN', 'LSTM']

for idx, (range_label, range_title) in enumerate(zip(range_order, range_titles)):
    ax = axes[idx]
    
    # Filter data for this range
    range_data = df_combined[df_combined['Range'] == range_label].copy()
    range_data['Model'] = pd.Categorical(range_data['Model'], categories=models_order, ordered=True)
    range_data = range_data.sort_values('Model')
    
    # Get data points count
    n_points = range_data['N'].iloc[0]
    
    # Create grouped bar chart with RMSE and MAE
    x = np.arange(len(models_order))
    width = 0.35
    
    rmse_values = range_data['RMSE'].values
    mae_values = range_data['MAE'].values
    
    # Create bars
    bars1 = ax.bar(x - width/2, rmse_values, width, label='RMSE', alpha=0.8)
    bars2 = ax.bar(x + width/2, mae_values, width, label='MAE', alpha=0.8)
    
    # Color bars based on ML/DL
    for i, model in enumerate(models_order):
        if model in ml_models:
            bars1[i].set_color('#3498db')
            bars2[i].set_color('#5dade2')
        else:
            bars1[i].set_color('#e74c3c')
            bars2[i].set_color('#ec7063')
    
    # Add value labels on bars (smaller font)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if not np.isnan(height):
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.2f}',
                       ha='center', va='bottom', fontsize=6.5)
    
    ax.set_xlabel('Model', fontsize=9, fontweight='bold')
    ax.set_ylabel('Error Metric', fontsize=9, fontweight='bold')
    ax.set_title(f'{range_title}\n(n = {n_points} test points)', 
                 fontsize=10, fontweight='bold', pad=8)
    ax.set_xticks(x)
    ax.set_xticklabels(models_order, rotation=0, fontsize=8)
    ax.legend(loc='upper center', fontsize=8, ncol=2)  # Changed to upper center
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)
    ax.tick_params(axis='y', labelsize=8)
    
    # Add horizontal line for reference
    ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    
    # Highlight best model (moved to upper right to avoid bars)
    best_idx = np.argmin(rmse_values)
    best_model = models_order[best_idx]
    best_rmse = rmse_values[best_idx]
    ax.text(0.98, 0.98, f'Best: {best_model}\nRMSE: {best_rmse:.3f}', 
            transform=ax.transAxes, fontsize=8, verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.6, pad=0.4))

plt.tight_layout(pad=2.0)
plt.savefig('error_analysis_by_range.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Figure 2 saved as 'error_analysis_by_range.png'")

# ============================================================
# FIGURE 3: COMPARATIVE PERFORMANCE TABLE (ELSEVIER OPTIMIZED)
# ============================================================

fig, ax = plt.subplots(figsize=(11, 6))  # Reduced from (14, 8)
ax.axis('tight')
ax.axis('off')

# Create summary table data
summary_data = []

for range_label in range_order:
    range_data = df_combined[df_combined['Range'] == range_label]
    
    # Best ML model
    ml_data = range_data[range_data['Model'].isin(ml_models)]
    best_ml_idx = ml_data['RMSE'].idxmin()
    best_ml = ml_data.loc[best_ml_idx]
    
    # Best DL model
    dl_data = range_data[range_data['Model'].isin(dl_models)]
    best_dl_idx = dl_data['RMSE'].idxmin()
    best_dl = dl_data.loc[best_dl_idx]
    
    summary_data.append([
        range_label.replace(' (0.25-4.0)', '').replace(' (4.0-8.0)', '').replace(' (8.0-11.0)', '').replace(' (11.0-13.0)', ''),
        f"{best_ml['N']} (ML)\n{best_dl['N']} (DL)",
        f"{best_ml['Model']}\n({best_ml['RMSE']:.3f})",
        f"{best_dl['Model']}\n({best_dl['RMSE']:.3f})",
        f"{best_ml['MAE']:.3f}\n{best_dl['MAE']:.3f}",
        f"{best_ml['ME']:+.3f}\n{best_dl['ME']:+.3f}"
    ])

# Create table
table = ax.table(cellText=summary_data,
                colLabels=['Range', 'N Points', 'Best ML\n(RMSE)', 'Best DL\n(RMSE)', 
                          'MAE\n(ML/DL)', 'Mean Error\n(ML/DL)'],
                cellLoc='center',
                loc='center',
                colWidths=[0.15, 0.12, 0.18, 0.18, 0.18, 0.19])

table.auto_set_font_size(False)
table.set_fontsize(9)  # Reduced from 10
table.scale(1, 2.5)  # Reduced from 3

# Style header
for i in range(6):
    cell = table[(0, i)]
    cell.set_facecolor('#34495e')
    cell.set_text_props(weight='bold', color='white', size=9)

# Style rows with alternating colors
for i in range(1, 5):
    for j in range(6):
        cell = table[(i, j)]
        cell.set_text_props(size=9)
        if i % 2 == 0:
            cell.set_facecolor('#ecf0f1')
        else:
            cell.set_facecolor('#ffffff')
        
        # Highlight best performers
        if j == 2:  # ML column
            cell.set_facecolor('#d5f4e6')
        elif j == 3:  # DL column
            cell.set_facecolor('#fadbd8')

plt.title('Summary: Best Model Performance by Rupture Life Range', 
          fontsize=12, fontweight='bold', pad=15)  # Reduced font size
plt.savefig('performance_summary_table.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Figure 3 saved as 'performance_summary_table.png'")

# ============================================================
# PRINT NUMERICAL SUMMARY FOR TEXT
# ============================================================

print("\n" + "="*80)
print("📊 SUMMARY STATISTICS FOR MANUSCRIPT")
print("="*80)

print("\n1. OVERALL PERFORMANCE:")
print(f"   ML Models - Mean RMSE: {df_combined[df_combined['Model'].isin(ml_models)]['RMSE'].mean():.4f}")
print(f"   DL Models - Mean RMSE: {df_combined[df_combined['Model'].isin(dl_models)]['RMSE'].mean():.4f}")

print("\n2. BEST OVERALL MODEL:")
best_overall = df_combined.groupby('Model')['RMSE'].mean().sort_values()
print(f"   {best_overall.index[0]}: {best_overall.values[0]:.4f} (average RMSE)")

print("\n3. WORST OVERALL MODEL:")
print(f"   {best_overall.index[-1]}: {best_overall.values[-1]:.4f} (average RMSE)")

print("\n4. RANGE-SPECIFIC CHALLENGES:")
for range_label in range_order:
    range_data = df_combined[df_combined['Range'] == range_label]
    avg_rmse = range_data['RMSE'].mean()
    print(f"   {range_label}: Average RMSE = {avg_rmse:.4f}")

print("\n5. MODELS WITH HIGHEST BIAS (|ME| > 0.5):")
high_bias = df_combined[df_combined['ME'].abs() > 0.5][['Model', 'Range', 'ME']].sort_values('ME', key=abs, ascending=False)
print(high_bias.to_string(index=False))

print("\n" + "="*80)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# CREATE COMPREHENSIVE ERROR ANALYSIS TABLE WITH INSIGHTS
# ============================================================

# Define ranges with actual hour conversions
range_info = [
    {
        'Range_Log': '0.25-4.0',
        'Range_Actual': '1.8-55 h',
        'Description': 'Very short-term',
        'Range_Label': 'Range 1 (0.25-4.0)'
    },
    {
        'Range_Log': '4.0-8.0',
        'Range_Actual': '55-3,000 h\n(2.3-125 days)',
        'Description': 'Short-to-medium',
        'Range_Label': 'Range 2 (4.0-8.0)'
    },
    {
        'Range_Log': '8.0-11.0',
        'Range_Actual': '3,000-60,000 h\n(125 days-6.8 years)',
        'Description': 'Long-term',
        'Range_Label': 'Range 3 (8.0-11.0)'
    },
    {
        'Range_Log': '11.0-13.0',
        'Range_Actual': '60,000-440,000 h\n(6.8-50 years)',
        'Description': 'Very long-term',
        'Range_Label': 'Range 4 (11.0-13.0)'
    }
]

# Prepare comprehensive table data
table_data = []

ml_models = ['LR', 'SVR', 'RF', 'GBR', 'XGB']
dl_models = ['DNN', 'BNN', 'CNN', 'LSTM']

for i, range_dict in enumerate(range_info):
    range_label = range_dict['Range_Label']
    range_data = df_combined[df_combined['Range'] == range_label]
    
    # Get best ML and DL models
    ml_data = range_data[range_data['Model'].isin(ml_models)]
    dl_data = range_data[range_data['Model'].isin(dl_models)]
    
    best_ml_idx = ml_data['RMSE'].idxmin()
    best_dl_idx = dl_data['RMSE'].idxmin()
    
    best_ml = ml_data.loc[best_ml_idx]
    best_dl = dl_data.loc[best_dl_idx]
    
    # Calculate average performance
    avg_ml_rmse = ml_data['RMSE'].mean()
    avg_dl_rmse = dl_data['RMSE'].mean()
    
    # Determine performance insight based on range
    if i == 0:  # Range 1 - Very short-term
        insight = "Sparse training data and extreme stress conditions cause all models to struggle. DL models severely over-predict due to extrapolation beyond training distribution. SVR's kernel-based approach provides better boundary handling."
    elif i == 1:  # Range 2 - Short-to-medium
        insight = "Moderate data availability enables good performance. Tree-based methods (GBR, XGB) excel due to effective handling of non-linear stress-temperature interactions. DL models show improvement but still lag ML."
    elif i == 2:  # Range 3 - Long-term
        insight = "Highest data concentration enables optimal performance for all models. DNN and SVR achieve comparable accuracy by effectively capturing complex creep mechanisms. This range represents typical service conditions."
    else:  # Range 4 - Very long-term
        insight = "Limited data and extrapolation requirements challenge all models. CNN catastrophically fails due to architectural mismatch. SVR and BNN maintain reasonable accuracy through regularization and uncertainty quantification."
    
    table_data.append({
        'Range_Log': range_dict['Range_Log'],
        'Range_Actual': range_dict['Range_Actual'],
        'Description': range_dict['Description'],
        'N_ML_DL': f"{best_ml['N']}/{best_dl['N']}",
        'Best_ML': f"{best_ml['Model']}\n(RMSE: {best_ml['RMSE']:.3f})",
        'Best_DL': f"{best_dl['Model']}\n(RMSE: {best_dl['RMSE']:.3f})",
        'Avg_RMSE': f"{avg_ml_rmse:.3f}/\n{avg_dl_rmse:.3f}",
        'Insight': insight
    })

df_table = pd.DataFrame(table_data)

# ============================================================
# CREATE PUBLICATION-QUALITY TABLE FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(18, 7))
ax.axis('tight')
ax.axis('off')

# Prepare table data for matplotlib
table_values = []
for _, row in df_table.iterrows():
    table_values.append([
        row['Range_Log'],
        row['Range_Actual'],
        row['Description'],
        row['N_ML_DL'],
        row['Best_ML'],
        row['Best_DL'],
        row['Avg_RMSE'],
        row['Insight']
    ])

# Create table
table = ax.table(
    cellText=table_values,
    colLabels=['Range\n(log hours)', 'Range\n(Actual hours)', 'Description', 
               'Sample\nSize', 'Best ML Model\n(RMSE)', 'Best DL Model\n(RMSE)', 
               'Average RMSE\n(ML/DL)', 'Performance Insight and Scientific Explanation'],
    cellLoc='left',
    loc='center',
    colWidths=[0.08, 0.12, 0.09, 0.06, 0.10, 0.10, 0.10, 0.35]
)

table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 3.5)

# Style header
for i in range(8):
    cell = table[(0, i)]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(weight='bold', color='white', ha='center')
    cell.set_height(0.08)

# Style data rows
colors = ['#ecf0f1', '#ffffff', '#ecf0f1', '#ffffff']
for i in range(1, 5):
    for j in range(8):
        cell = table[(i, j)]
        cell.set_facecolor(colors[i-1])
        cell.set_text_props(va='top', wrap=True)
        cell.set_height(0.15)
        
        # Center alignment for numeric columns
        if j in [0, 3, 4, 5, 6]:
            cell.set_text_props(ha='center', va='center')
        
        # Highlight best performers
        if j == 4:  # Best ML column
            cell.set_facecolor('#d5f4e6')
        elif j == 5:  # Best DL column
            cell.set_facecolor('#fadbd8')

# Add borders
for key, cell in table.get_celld().items():
    cell.set_edgecolor('gray')
    cell.set_linewidth(0.5)

plt.title('Comprehensive Error Distribution Analysis Across Rupture Life Ranges', 
          fontsize=13, fontweight='bold', pad=20)

plt.savefig('comprehensive_error_analysis_table.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Comprehensive table saved as 'comprehensive_error_analysis_table.png'")

# ============================================================
# ALSO CREATE A LATEX TABLE VERSION FOR DIRECT MANUSCRIPT USE
# ============================================================

print("\n" + "="*120)
print("📋 LATEX TABLE CODE FOR MANUSCRIPT")
print("="*120)

latex_table = """
\\begin{table*}[ht]
\\centering
\\caption{Comprehensive Error Distribution Analysis Across Rupture Life Ranges}
\\label{tab:error_analysis}
\\small
\\begin{tabular}{p{1.2cm}p{2cm}p{1.5cm}p{1cm}p{2cm}p{2cm}p{1.5cm}p{6cm}}
\\hline
\\textbf{Range (log)} & \\textbf{Range (Actual)} & \\textbf{Description} & \\textbf{N} & \\textbf{Best ML} & \\textbf{Best DL} & \\textbf{Avg RMSE} & \\textbf{Performance Insight} \\\\
\\hline
"""

for _, row in df_table.iterrows():
    latex_table += f"{row['Range_Log']} & {row['Range_Actual'].replace(chr(10), ' ')} & {row['Description']} & {row['N_ML_DL']} & "
    latex_table += f"{row['Best_ML'].replace(chr(10), ' ')} & {row['Best_DL'].replace(chr(10), ' ')} & "
    latex_table += f"{row['Avg_RMSE'].replace(chr(10), '/')} & "
    latex_table += f"{row['Insight']} \\\\\n"

latex_table += """\\hline
\\end{tabular}
\\end{table*}
"""

print(latex_table)

# ============================================================
# EXPORT TO EXCEL FOR EASY EDITING
# ============================================================

df_table.to_excel('comprehensive_error_analysis_table.xlsx', index=False)
print("\n✅ Table also exported to Excel: 'comprehensive_error_analysis_table.xlsx'")

print("\n" + "="*120)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# CREATE COMPREHENSIVE ERROR ANALYSIS TABLE WITH INSIGHTS
# ============================================================

# Define ranges with actual hour conversions
range_info = [
    {
        'Range_Log': '0.25-4.0',
        'Range_Actual': '1.8-55 h',
        'Description': 'Very short-term',
        'Range_Label': 'Range 1 (0.25-4.0)'
    },
    {
        'Range_Log': '4.0-8.0',
        'Range_Actual': '55-3,000 h\n(2.3-125 days)',
        'Description': 'Short-to-medium',
        'Range_Label': 'Range 2 (4.0-8.0)'
    },
    {
        'Range_Log': '8.0-11.0',
        'Range_Actual': '3,000-60,000 h\n(125 days-6.8 years)',
        'Description': 'Long-term',
        'Range_Label': 'Range 3 (8.0-11.0)'
    },
    {
        'Range_Log': '11.0-13.0',
        'Range_Actual': '60,000-440,000 h\n(6.8-50 years)',
        'Description': 'Very long-term',
        'Range_Label': 'Range 4 (11.0-13.0)'
    }
]

# Prepare comprehensive table data
table_data = []

ml_models = ['LR', 'SVR', 'RF', 'GBR', 'XGB']
dl_models = ['DNN', 'BNN', 'CNN', 'LSTM']

for i, range_dict in enumerate(range_info):
    range_label = range_dict['Range_Label']
    range_data = df_combined[df_combined['Range'] == range_label]
    
    # Get best ML and DL models
    ml_data = range_data[range_data['Model'].isin(ml_models)]
    dl_data = range_data[range_data['Model'].isin(dl_models)]
    
    best_ml_idx = ml_data['RMSE'].idxmin()
    best_dl_idx = dl_data['RMSE'].idxmin()
    
    best_ml = ml_data.loc[best_ml_idx]
    best_dl = dl_data.loc[best_dl_idx]
    
    # Calculate average performance
    avg_ml_rmse = ml_data['RMSE'].mean()
    avg_dl_rmse = dl_data['RMSE'].mean()
    avg_ml_mae = ml_data['MAE'].mean()
    avg_dl_mae = dl_data['MAE'].mean()
    
    # Determine performance insight based on range
    if i == 0:  # Range 1 - Very short-term
        insight = "Sparse training data and extreme stress conditions cause all models to struggle. DL models severely over-predict due to extrapolation beyond training distribution. SVR's kernel-based approach provides better boundary handling."
    elif i == 1:  # Range 2 - Short-to-medium
        insight = "Moderate data availability enables good performance. Tree-based methods (GBR, XGB) excel due to effective handling of non-linear stress-temperature interactions. DL models show improvement but still lag ML."
    elif i == 2:  # Range 3 - Long-term
        insight = "Highest data concentration enables optimal performance for all models. DNN and SVR achieve comparable accuracy by effectively capturing complex creep mechanisms. This range represents typical service conditions."
    else:  # Range 4 - Very long-term
        insight = "Limited data and extrapolation requirements challenge all models. CNN catastrophically fails due to architectural mismatch. SVR and BNN maintain reasonable accuracy through regularization and uncertainty quantification."
    
    table_data.append({
        'Range_Log': range_dict['Range_Log'],
        'Range_Actual': range_dict['Range_Actual'],
        'Description': range_dict['Description'],
        'N_ML_DL': f"{best_ml['N']}/{best_dl['N']}",
        'Best_ML': f"{best_ml['Model']}\nRMSE: {best_ml['RMSE']:.3f}\nMAE: {best_ml['MAE']:.3f}",
        'Best_DL': f"{best_dl['Model']}\nRMSE: {best_dl['RMSE']:.3f}\nMAE: {best_dl['MAE']:.3f}",
        'Avg_Errors': f"ML: {avg_ml_rmse:.3f}/{avg_ml_mae:.3f}\nDL: {avg_dl_rmse:.3f}/{avg_dl_mae:.3f}",
        'Insight': insight
    })

df_table = pd.DataFrame(table_data)

# ============================================================
# CREATE PUBLICATION-QUALITY TABLE FIGURE
# ============================================================

fig, ax = plt.subplots(figsize=(18, 7))
ax.axis('tight')
ax.axis('off')

# Prepare table data for matplotlib
table_values = []
for _, row in df_table.iterrows():
    table_values.append([
        row['Range_Log'],
        row['Range_Actual'],
        row['Description'],
        row['N_ML_DL'],
        row['Best_ML'],
        row['Best_DL'],
        row['Avg_Errors'],
        row['Insight']
    ])

# Create table
table = ax.table(
    cellText=table_values,
    colLabels=['Range\n(log hours)', 'Range\n(Actual hours)', 'Description', 
               'Sample\nSize\n(ML/DL)', 'Best ML Model\n(RMSE/MAE)', 'Best DL Model\n(RMSE/MAE)', 
               'Average Errors\n(RMSE/MAE)', 'Performance Insight and Scientific Explanation'],
    cellLoc='left',
    loc='center',
    colWidths=[0.07, 0.11, 0.08, 0.06, 0.11, 0.11, 0.11, 0.35]
)

table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 3.5)

# Style header
for i in range(8):
    cell = table[(0, i)]
    cell.set_facecolor('#2c3e50')
    cell.set_text_props(weight='bold', color='white', ha='center')
    cell.set_height(0.08)

# Style data rows
colors = ['#ecf0f1', '#ffffff', '#ecf0f1', '#ffffff']
for i in range(1, 5):
    for j in range(8):
        cell = table[(i, j)]
        cell.set_facecolor(colors[i-1])
        cell.set_text_props(va='top', wrap=True, size=8.5)
        cell.set_height(0.15)
        
        # Center alignment for numeric columns
        if j in [0, 3, 4, 5, 6]:
            cell.set_text_props(ha='center', va='center', size=8.5)
        
        # Highlight best performers
        if j == 4:  # Best ML column
            cell.set_facecolor('#d5f4e6')
        elif j == 5:  # Best DL column
            cell.set_facecolor('#fadbd8')

# Add borders
for key, cell in table.get_celld().items():
    cell.set_edgecolor('gray')
    cell.set_linewidth(0.5)

plt.title('Comprehensive Error Distribution Analysis Across Rupture Life Ranges', 
          fontsize=13, fontweight='bold', pad=20)

plt.savefig('comprehensive_error_analysis_table.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Comprehensive table saved as 'comprehensive_error_analysis_table.png'")

# ============================================================
# ALSO CREATE A LATEX TABLE VERSION FOR DIRECT MANUSCRIPT USE
# ============================================================

print("\n" + "="*120)
print("📋 LATEX TABLE CODE FOR MANUSCRIPT")
print("="*120)

latex_table = """
\\begin{table*}[ht]
\\centering
\\caption{Comprehensive Error Distribution Analysis Across Rupture Life Ranges}
\\label{tab:error_analysis}
\\small
\\begin{tabular}{p{1cm}p{2cm}p{1.3cm}p{0.9cm}p{1.8cm}p{1.8cm}p{1.5cm}p{6cm}}
\\hline
\\textbf{Range (log)} & \\textbf{Range (Actual)} & \\textbf{Description} & \\textbf{N} & \\textbf{Best ML (RMSE/MAE)} & \\textbf{Best DL (RMSE/MAE)} & \\textbf{Avg Errors} & \\textbf{Performance Insight} \\\\
\\hline
"""

for _, row in df_table.iterrows():
    latex_table += f"{row['Range_Log']} & {row['Range_Actual'].replace(chr(10), ' ')} & {row['Description']} & {row['N_ML_DL']} & "
    latex_table += f"{row['Best_ML'].replace(chr(10), ' ')} & {row['Best_DL'].replace(chr(10), ' ')} & "
    latex_table += f"{row['Avg_Errors'].replace(chr(10), ' ')} & "
    latex_table += f"{row['Insight']} \\\\\n"

latex_table += """\\hline
\\end{tabular}
\\end{table*}
"""

print(latex_table)

# ============================================================
# EXPORT TO EXCEL FOR EASY EDITING
# ============================================================

df_table.to_excel('comprehensive_error_analysis_table.xlsx', index=False)
print("\n✅ Table also exported to Excel: 'comprehensive_error_analysis_table.xlsx'")

print("\n" + "="*120)

# ============================================================
# PRINT TABLE PREVIEW
# ============================================================

print("\n" + "="*120)
print("📊 TABLE PREVIEW")
print("="*120)
print(df_table.to_string(index=False))
print("="*120)